In [ ]:
from dgl import DGLGraph
import pandas as pd
from rdkit.Chem import MolFromSmiles
import numpy as np
from rdkit import Chem
import torch as th
from dgl.data.graph_serialize import save_graphs, load_graphs, load_labels
import torch
import rdkit
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

def find_var(df, min_value):
    df1 = df.copy()
    df1.loc["var"] = np.var(df1.values, axis=0)
    col = [x for i, x in enumerate(df1.columns)if df1.iat[-1, i] < min_value]
    return col


def find_sum_0(df):
    '''
    input: df
    return: the columns of labels with no positive labels
    '''
    df1 = df.copy()
    df1.loc["sum"] = np.sum(df1.values, axis=0)
    col = [x for i, x in enumerate(df1.columns)if df1.iat[-1, i] == 0]
    return col


def find_sum_1(df):
    '''
    input: df
    return: the columns of labels with no negative labels
    '''
    df1 = df.copy()
    df1.loc["sum"] = np.sum(df1.values, axis=0)
    col = [x for i, x in enumerate(df1.columns)if df1.iat[-1, i] == len(df)]
    return col


def one_of_k_encoding(x, allowable_set):
    if x not in allowable_set:
        raise Exception("input {0} not in allowable set{1}:".format(
            x, allowable_set))
    return [x == s for s in allowable_set]


def one_of_k_encoding_unk(x, allowable_set):
    """Maps inputs not in the allowable set to the last element."""
    if x not in allowable_set:
        x = allowable_set[-1]
    return [x == s for s in allowable_set]


def atom_features(atom, explicit_H = False, use_chirality=True):
    results = one_of_k_encoding_unk(
      atom.GetSymbol(),
      [
        'B',
        'C',
        'N',
        'O',
        'F',
        'Si',
        'P',
        'S',
        'Cl',
        'As',
        'Se',
        'Br',
        'Te',
        'I',
        'At',
        'other'
      ]) + one_of_k_encoding(atom.GetDegree(),
                             [0, 1, 2, 3, 4, 5, 6]) + \
              [atom.GetFormalCharge(), atom.GetNumRadicalElectrons()] + \
              one_of_k_encoding_unk(atom.GetHybridization(), [
                Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
                Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
                Chem.rdchem.HybridizationType.SP3D2,'other']) + [atom.GetIsAromatic()]
                # [atom.GetIsAromatic()] # set all aromaticity feature blank.
    # In case of explicit hydrogen(QM8, QM9), avoid calling `GetTotalNumHs`
    if not explicit_H:
        results = results + one_of_k_encoding_unk(atom.GetTotalNumHs(),
                                                  [0, 1, 2, 3, 4])
    if use_chirality:
        try:
            results = results + one_of_k_encoding_unk(
                atom.GetProp('_CIPCode'),
                ['R', 'S']) + [atom.HasProp('_ChiralityPossible')]
        except:
            results = results + [False, False
                                 ] + [atom.HasProp('_ChiralityPossible')]

    return np.array(results)


def one_of_k_atompair_encoding(x, allowable_set):
    for atompair in allowable_set:
        if x in atompair:
            x = atompair
            break
        else:
            if atompair == allowable_set[-1]:
                x = allowable_set[-1]
            else:
                continue
    return [x == s for s in allowable_set]


def bond_features(bond, use_chirality=True, atompair=False):
    bt = bond.GetBondType()
    bond_feats = [
        bt == Chem.rdchem.BondType.SINGLE, bt == Chem.rdchem.BondType.DOUBLE,
        bt == Chem.rdchem.BondType.TRIPLE, bt == Chem.rdchem.BondType.AROMATIC,
        bond.GetIsConjugated(),
        bond.IsInRing()
    ]
    if use_chirality:
        bond_feats = bond_feats + one_of_k_encoding_unk(
            str(bond.GetStereo()),
            ["STEREONONE", "STEREOANY", "STEREOZ", "STEREOE"])
    if atompair:
        atom_pair_str = bond.GetBeginAtom().GetSymbol() + bond.GetEndAtom().GetSymbol()
        bond_feats = bond_feats + one_of_k_atompair_encoding(
            atom_pair_str, [['CC'], ['CN', 'NC'], ['ON', 'NO'], ['CO', 'OC'], ['CS', 'SC'],
                            ['SO', 'OS'], ['NN'], ['SN', 'NS'], ['CCl', 'ClC'], ['CF', 'FC'],
                            ['CBr', 'BrC'], ['others']]
        )

    return np.array(bond_feats).astype(int)


def etype_features(bond, use_chirality=True, atompair=True):
    bt = bond.GetBondType()
    bond_feats_1 = [
        bt == Chem.rdchem.BondType.SINGLE, bt == Chem.rdchem.BondType.DOUBLE,
        bt == Chem.rdchem.BondType.TRIPLE, bt == Chem.rdchem.BondType.AROMATIC,
    ]
    for i, m in enumerate(bond_feats_1):
        if m == True:
            a = i

    bond_feats_2 = bond.GetIsConjugated()
    if bond_feats_2 == True:
        b = 1
    else:
        b = 0

    bond_feats_3 = bond.IsInRing
    if bond_feats_3 == True:
        c = 1
    else:
        c = 0

    index = a * 1 + b * 4 + c * 8
    if use_chirality:
        bond_feats_4 = one_of_k_encoding_unk(
            str(bond.GetStereo()),
            ["STEREONONE", "STEREOANY", "STEREOZ", "STEREOE"])
        for i, m in enumerate(bond_feats_4):
            if m == True:
                d = i
        index = index + d * 16
    if atompair == True:
        atom_pair_str = bond.GetBeginAtom().GetSymbol() + bond.GetEndAtom().GetSymbol()
        bond_feats_5 = one_of_k_atompair_encoding(
            atom_pair_str, [['CC'], ['CN', 'NC'], ['ON', 'NO'], ['CO', 'OC'], ['CS', 'SC'],
                            ['SO', 'OS'], ['NN'], ['SN', 'NS'], ['CCl', 'ClC'], ['CF', 'FC'],
                            ['CBr', 'BrC'], ['others']]
        )
        for i, m in enumerate(bond_feats_5):
            if m == True:
                e = i
        index = index + e*64
    return index


def construct_RGCN_bigraph_from_smiles(smiles):
    g = DGLGraph()

    # Add nodes
    mol = MolFromSmiles(smiles)
    num_atoms = mol.GetNumAtoms()
    g.add_nodes(num_atoms)
    atoms_feature_all = []
    for atom_index, atom in enumerate(mol.GetAtoms()):
        atom_feature = atom_features(atom).tolist()
        atoms_feature_all.append(atom_feature)
    g.ndata["atom"] = torch.tensor(atoms_feature_all)



    # Add edges
    src_list = []
    dst_list = []
    etype_feature_all = []
    num_bonds = mol.GetNumBonds()
    for i in range(num_bonds):
        bond = mol.GetBondWithIdx(i)
        etype_feature = etype_features(bond)
        u = bond.GetBeginAtomIdx()
        v = bond.GetEndAtomIdx()
        src_list.extend([u, v])
        dst_list.extend([v, u])
        etype_feature_all.append(etype_feature)
        etype_feature_all.append(etype_feature)

    g.add_edges(src_list, dst_list)
    normal_all = []
    for i in etype_feature_all:
        normal = etype_feature_all.count(i)/len(etype_feature_all)
        normal = round(normal, 1)
        normal_all.append(normal)

    g.edata["etype"] = torch.tensor(etype_feature_all)
    g.edata["normal"] = torch.tensor(normal_all)
    return g


def binary_class_split(dataset, label_name):
    zero_dataset = []
    one_dataset = []
    for i in range(len(dataset)):
        if dataset[i][label_name] == 0:
            zero_dataset.append(dataset[i])
        else:
            one_dataset.append(dataset[i])
    return zero_dataset, one_dataset


def data_set_random_split(Dataset, shuffle=False):
    if shuffle:
        np.random.shuffle(Dataset)
    train_seq    = [i for i in range(len(Dataset)) if not i % 5 == 0]
    evaluate_seq = [i for i in range(len(Dataset)) if i % 5 == 0]
    training_set = []
    val_set = []
    test_set = []
    for i in train_seq:
        training_set.append(Dataset[i])
    for i in evaluate_seq:
        if i % 2 == 0:
            val_set.append(Dataset[i])
        else:
            test_set.append(Dataset[i])
    return training_set, val_set, test_set


def get_0_1_index(df, labels):
    col_zero = [i for i, x in enumerate(df[labels]) if x == 0]
    col_one = [i for i, x in enumerate(df[labels]) if x == 1]
    return col_zero, col_one


def split_dataset(dataset):
    train_seq = [x for x in range(len(dataset)) if not x % 5 == 0]
    evaluate_seq = [x for x in range(len(dataset)) if x % 5 == 0]
    val_seq = [x for x in evaluate_seq if not x % 2 == 0]
    test_seq = [x for x in evaluate_seq if x % 2 == 0]
    train_set = dataset.loc[train_seq].reset_index(drop=True)
    val_set = dataset.loc[val_seq].reset_index(drop=True)
    test_set = dataset.loc[test_seq].reset_index(drop=True)
    return train_set, val_set, test_set


def build_dataset(dataset_smiles, label_name, smiles_name, descriptor_seq, is_descriptor=True):
    dataset_gnn = []
    labels = dataset_smiles[label_name]
    smilesList = dataset_smiles[smiles_name]
    for i, smiles in enumerate(smilesList):
        if np.isnan(labels[i]):
            continue
        try:
            g = construct_RGCN_bigraph_from_smiles(smiles)
            if is_descriptor:
                molecule = [smiles, g, [labels[i]], dataset_smiles.loc[i][descriptor_seq], build_mask(labels)]
            else:
                molecule = [smiles, g, [labels[i]], build_mask(labels)]
            dataset_gnn.append(molecule)
        except:
            print('{} is transformed failed!'.format(smiles))
    return dataset_gnn


def build_mask(labels_list, mask_value=100):
    mask = []
    for i in labels_list:
        if i==mask_value:
            mask.append(0)
        else:
            mask.append(1)
    return mask


def multi_task_build_dataset(dataset_smiles, labels_list, smiles_name):
    dataset_gnn = []
    failed_molecule = []
    labels = dataset_smiles[labels_list]
    split_index = dataset_smiles['group']
    smilesList = dataset_smiles[smiles_name]
    molecule_number = len(smilesList)
    for i, smiles in enumerate(smilesList):
        try:
            g = construct_RGCN_bigraph_from_smiles(smiles)
            mask = build_mask(labels.loc[i], mask_value=123456)
            molecule = [smiles, g, labels.loc[i], mask, split_index.loc[i]]
            dataset_gnn.append(molecule)
            print('{}/{} molecule is transformed!'.format(i+1, molecule_number))
        except:
            print('{} is transformed failed!'.format(smiles))
            molecule_number = molecule_number - 1
            failed_molecule.append(smiles)
    print('{}({}) is transformed failed!'.format(failed_molecule, len(failed_molecule)))
    return dataset_gnn


def built_data_and_save_for_splited(
        origin_path='example.csv',
        save_path='example.bin',
        group_path='example_group.csv',
        task_list_selected=None,
         ):
    '''
        origin_path: str
            origin csv data set path, including molecule name, smiles, task
        save_path: str
            graph out put path
        group_path: str
            group out put path
        task_list_selected: list
            a list of selected task
        '''
    data_origin = pd.read_csv(origin_path)
    smiles_name = 'smiles'
    data_origin = data_origin.fillna(123456)
    labels_list = [x for x in data_origin.columns if x not in ['smiles', 'group']]
    if task_list_selected is not None:
        labels_list = task_list_selected
    data_set_gnn = multi_task_build_dataset(dataset_smiles=data_origin, labels_list=labels_list, smiles_name=smiles_name)

    smiles, graphs, labels, mask, split_index = map(list, zip(*data_set_gnn))
    graph_labels = {'labels': torch.tensor(labels),
                    'mask': torch.tensor(mask)
                    }
    split_index_pd = pd.DataFrame(columns=['smiles', 'group'])
    split_index_pd.smiles = smiles
    split_index_pd.group = split_index
    split_index_pd.to_csv(group_path, index=None, columns=None)
    print('Molecules graph is saved!')
    save_graphs(save_path, graphs, graph_labels)


def standardization_np(data, mean, std):
    return (data - mean) / (std + 1e-10)


def re_standar_np(data, mean, std):
    return data*(std + 1e-10)+mean
    

def split_dataset_according_index(dataset, train_index, val_index, test_index, data_type='np'):
    if data_type =='pd':
        return pd.DataFrame(dataset[train_index]), pd.DataFrame(dataset[val_index]), pd.DataFrame(dataset[test_index])
    if data_type =='np':
        return dataset[train_index], dataset[val_index], dataset[test_index]


def load_graph_from_csv_bin_for_splited(
        bin_path='example.bin',
        group_path='example.csv',
        select_task_index=None):
    smiles = pd.read_csv(group_path, index_col=None).smiles.values
    group = pd.read_csv(group_path, index_col=None).group.to_list()
    graphs, detailed_information = load_graphs(bin_path)
    labels = detailed_information['labels']
    mask = detailed_information['mask']
    if select_task_index is not None:
        labels = labels[:, select_task_index]
        mask = mask[:, select_task_index]
    # calculate not_use index
    notuse_mask = torch.mean(mask.float(), 1).numpy().tolist()
    not_use_index = []
    for index, notuse in enumerate(notuse_mask):
        if notuse==0:
            not_use_index.append(index)
    train_index=[]
    val_index = []
    test_index = []
    for index, group_index in enumerate(group):
        if group_index=='training' and index not in not_use_index:
            train_index.append(index)
        if group_index=='valid' and index not in not_use_index:
            val_index.append(index)
        if group_index == 'test' and index not in not_use_index:
            test_index.append(index)
    graph_List = []
    for g in graphs:
        graph_List.append(g)
    graphs_np = np.array(graphs)
    train_smiles, val_smiles, test_smiles = split_dataset_according_index(smiles, train_index, val_index, test_index)
    train_labels, val_labels, test_labels = split_dataset_according_index(labels.numpy(), train_index, val_index,
                                                                          test_index, data_type='pd')
    train_mask, val_mask, test_mask = split_dataset_according_index(mask.numpy(), train_index, val_index, test_index,
                                                                    data_type='pd')
    train_graph, val_graph, test_graph = split_dataset_according_index(graphs_np, train_index, val_index, test_index)

    # delete the 0_pos_label and 0_neg_label
    task_number = train_labels.values.shape[1]

    train_set = []
    val_set = []
    test_set = []
    for i in range(len(train_index)):
        molecule = [train_smiles[i], train_graph[i], train_labels.values[i], train_mask.values[i]]
        train_set.append(molecule)

    for i in range(len(val_index)):
        molecule = [val_smiles[i], val_graph[i], val_labels.values[i], val_mask.values[i]]
        val_set.append(molecule)

    for i in range(len(test_index)):
        molecule = [test_smiles[i], test_graph[i], test_labels.values[i], test_mask.values[i]]
        test_set.append(molecule)
    print(len(train_set), len(val_set), len(test_set),  task_number)
    return train_set, val_set, test_set, task_number

def construct_RGCN_bigraph_from_smiles_with_species_gender_exposure(smiles, D55,D74, bins):
    g = DGLGraph()

    # Add nodes
    mol = MolFromSmiles(smiles)
    num_atoms = mol.GetNumAtoms()
    g.add_nodes(num_atoms)
    atoms_feature_all = []
    D55_feature_all = []
    D74_feature_all = []
    bins_feature_all = []
   
    bins_feature = [0]*3
    bins_feature[bins] = 1
  
    D55_feature = [D55]
    D74_feature = [D74]
    
    for atom_index, atom in enumerate(mol.GetAtoms()):
        atom_feature = atom_features(atom).tolist()
        atoms_feature_all.append(atom_feature)
        D55_feature_all.append(D55_feature)
        D74_feature_all.append(D74_feature)
        bins_feature_all.append(bins_feature)
       
        
    atoms_tensor = torch.tensor(atoms_feature_all)
    D55_tensor = torch.tensor(D55_feature_all)
    D74_tensor = torch.tensor(D74_feature_all)
    bins_tensor = torch.tensor(bins_feature_all)
    

    node_features = torch.cat((atoms_tensor, D55_tensor, D74_tensor, bins_tensor), dim=1)
    g.ndata["atom"] = node_features


    # Add edges
    src_list = []
    dst_list = []
    etype_feature_all = []
    num_bonds = mol.GetNumBonds()
    for i in range(num_bonds):
        bond = mol.GetBondWithIdx(i)
        etype_feature = etype_features(bond)
        u = bond.GetBeginAtomIdx()
        v = bond.GetEndAtomIdx()
        src_list.extend([u, v])
        dst_list.extend([v, u])
        etype_feature_all.append(etype_feature)
        etype_feature_all.append(etype_feature)

    g.add_edges(src_list, dst_list)
    normal_all = []
    for i in etype_feature_all:
        normal = etype_feature_all.count(i)/len(etype_feature_all)
        normal = round(normal, 1)
        normal_all.append(normal)

    g.edata["etype"] = torch.tensor(etype_feature_all)
    g.edata["normal"] = torch.tensor(normal_all)
    return g

def multi_task_build_dataset_with_species_gender_exposure(dataset_smiles, labels_list, smiles_name, D55_list, D74_list, 
                                                          bins_list):
    dataset_gnn = []
    failed_molecule = []
    labels = dataset_smiles[labels_list]
    split_index = dataset_smiles['group']
    smilesList = dataset_smiles[smiles_name]
    molecule_number = len(smilesList)
    for i, smiles in enumerate(smilesList):
        try:
            g = construct_RGCN_bigraph_from_smiles_with_species_gender_exposure(smiles, D55_list[i], D74_list[i], 
                                                                                bins_list[i])
            mask = build_mask(labels.loc[i], mask_value=123456)
            molecule = [smiles, g, labels.loc[i], mask, split_index.loc[i]]
            dataset_gnn.append(molecule)
            print('{}/{} molecule is transformed!'.format(i+1, molecule_number))
        except:
            print('{} is transformed failed!'.format(smiles))
            molecule_number = molecule_number - 1
            failed_molecule.append(smiles)
    print('{}({}) is transformed failed!'.format(failed_molecule, len(failed_molecule)))
    return dataset_gnn


def built_data_and_save_for_splited_with_species_gender_exposure(
        origin_path='example.csv',
        save_path='example.bin',
        group_path='example_group.csv',
        task_list_selected=None,
        D55_name='D55',
        D74_name='D74',
        bins_name='bins',
       
         ):
    '''
        origin_path: str
            origin csv data set path, including molecule name, smiles, task
        save_path: str
            graph out put path
        group_path: str
            group out put path
        task_list_selected: list
            a list of selected task
        species_name: str
            the name of species column
        gender_name: str
            the name of gender column
        exposure_name: str
            the name of exposure column
        '''
    data_origin = pd.read_csv(origin_path)
    smiles_name = 'smiles'
    data_origin = data_origin.fillna(123456)
    labels_list = [x for x in data_origin.columns if x not in ['smiles', 'group', D55_name, D74_name,bins_name]]
    if task_list_selected is not None:
        labels_list = task_list_selected
    data_set_gnn = multi_task_build_dataset_with_species_gender_exposure(dataset_smiles=data_origin, labels_list=labels_list, 
                                                                         smiles_name=smiles_name, 
                                                                         D55_list=data_origin[D55_name], 
                                                                         D74_list=data_origin[D74_name], 
                                                                         bins_list=data_origin[bins_name]
                                                                        )

    smiles, graphs, labels, mask, split_index = map(list, zip(*data_set_gnn))
    graph_labels = {'labels': torch.tensor(labels),
                    'mask': torch.tensor(mask)
                    }
    split_index_pd = pd.DataFrame(columns=['smiles', 'group'])
    split_index_pd.smiles = smiles
    split_index_pd.group = split_index
    split_index_pd.to_csv(group_path, index=None, columns=None)
    print('Molecules graph is saved!')
    save_graphs(save_path, graphs, graph_labels)

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D, MolToFile
import matplotlib.cm as cm
import matplotlib
from IPython.display import SVG, display
import seaborn as sns

sns.set(color_codes=True)


def _normalize_atom_weights(atom_weight):
    """Normalize atom weights to the [0, 1] range for continuous visualization."""
    if hasattr(atom_weight, "cpu"):
        weights = atom_weight.squeeze().cpu().numpy()
    else:
        weights = np.asarray(atom_weight).squeeze()

    if np.ndim(weights) == 0:
        weights = np.array([float(weights)], dtype=float)
    else:
        weights = np.asarray(weights, dtype=float).flatten()

    if len(weights) == 0:
        return weights

    weight_min = float(np.min(weights))
    weight_max = float(np.max(weights))
    if weight_max > weight_min:
        weights = (weights - weight_min) / (weight_max - weight_min)
    else:
        weights = np.full_like(weights, 0.5, dtype=float)
    return weights


def weight_visualize_continuous(smiles, atom_weight, save_path=None, size=(280, 280), show=True):
    """Visualize atom weights using continuous color mapping only."""
    print(smiles)
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"Warning: unable to create a molecule from SMILES '{smiles}'")
        return None

    atom_new_weight = _normalize_atom_weights(atom_weight)
    if len(atom_new_weight) != mol.GetNumAtoms():
        print(
            f"Warning: the number of atom weights ({len(atom_new_weight)}) does not match "
            f"the number of atoms ({mol.GetNumAtoms()}); using a neutral weight of 0.5."
        )
        atom_new_weight = np.full(mol.GetNumAtoms(), 0.5, dtype=float)

    norm = matplotlib.colors.Normalize(vmin=0, vmax=1)
    cmap = cm.get_cmap('Oranges')
    plt_colors = cm.ScalarMappable(norm=norm, cmap=cmap)

    atom_colors = {i: plt_colors.to_rgba(float(atom_new_weight[i])) for i in range(mol.GetNumAtoms())}
    bond_colors = {}
    for i in range(mol.GetNumBonds()):
        bond = mol.GetBondWithIdx(i)
        u = bond.GetBeginAtomIdx()
        v = bond.GetEndAtomIdx()
        bond_weight = (atom_new_weight[u] + atom_new_weight[v]) / 2
        bond_colors[i] = plt_colors.to_rgba(float(abs(bond_weight)))

    rdDepictor.Compute2DCoords(mol)
    drawer = rdMolDraw2D.MolDraw2DSVG(size[0], size[1])
    drawer.SetFontSize(1)
    mol = rdMolDraw2D.PrepareMolForDrawing(mol)
    drawer.DrawMolecule(
        mol,
        highlightAtoms=list(range(mol.GetNumAtoms())),
        highlightBonds=list(range(mol.GetNumBonds())),
        highlightAtomColors=atom_colors,
        highlightBondColors=bond_colors,
    )
    drawer.FinishDrawing()

    svg = drawer.GetDrawingText().replace('svg:', '')
    if save_path is not None:
        svg_path = f"{save_path}.svg" if not str(save_path).endswith('.svg') else str(save_path)
        with open(svg_path, 'w', encoding='utf-8') as f:
            f.write(svg)
        print(f"Atom-weight visualization saved to: {svg_path}")
    else:
        svg_path = None

    if show:
        display(SVG(svg))
    return svg_path


In [ ]:
import datetime
from sklearn.metrics import roc_auc_score, mean_squared_error, precision_recall_curve, auc, r2_score
import torch
import torch.nn.functional as F
import dgl
import numpy as np
import random
from dgl.readout import sum_nodes
from dgl.nn.pytorch.conv import RelGraphConv
from torch import nn
import pandas as pd



class WeightAndSum(nn.Module):
    def __init__(self, in_feats, task_num=1, attention=True, return_weight=False):
        super(WeightAndSum, self).__init__()
        self.attention = attention
        self.in_feats = in_feats
        self.task_num = task_num
        self.return_weight=return_weight
        self.atom_weighting_specific = nn.ModuleList([self.atom_weight(self.in_feats) for _ in range(self.task_num)])
        self.shared_weighting = self.atom_weight(self.in_feats)
    def forward(self, bg, feats):
        feat_list = []
        atom_list = []
        # cal specific feats
        for i in range(self.task_num):
            with bg.local_scope():
                bg.ndata['h'] = feats
                weight = self.atom_weighting_specific[i](feats)
                bg.ndata['w'] = weight
                specific_feats_sum = sum_nodes(bg, 'h', 'w')
                atom_list.append(bg.ndata['w'])
            feat_list.append(specific_feats_sum)

        # cal shared feats
        with bg.local_scope():
            bg.ndata['h'] = feats
            bg.ndata['w'] = self.shared_weighting(feats)
            shared_feats_sum = sum_nodes(bg, 'h', 'w')
        # feat_list.append(shared_feats_sum)
        if self.attention:
            if self.return_weight:
                return feat_list, atom_list
            else:
                return feat_list
        else:
            return shared_feats_sum

    def atom_weight(self, in_feats):
        return nn.Sequential(
            nn.Linear(in_feats, 1),
            nn.Sigmoid()
            )


class MLPBinaryClassifier(nn.Module):
    def __init__(self, in_feats, hidden_feats, n_tasks, dropout=0.):
        super(MLPBinaryClassifier, self).__init__()

        self.predict = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_feats, hidden_feats),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_feats),
            nn.Linear(hidden_feats, n_tasks)
        )

    def forward(self, h):
        return self.predict(h)


class RGCNLayer(nn.Module):
    def __init__(self, in_feats, out_feats, num_rels=64*21, activation=F.relu, loop=False,
                 residual=True, batchnorm=True, rgcn_drop_out=0.5):
        super(RGCNLayer, self).__init__()

        self.activation = activation
        self.graph_conv_layer = RelGraphConv(in_feats, out_feats, num_rels=num_rels, regularizer='basis',
                                               num_bases=None, bias=True, activation=activation,
                                               self_loop=loop, dropout=rgcn_drop_out)
        self.residual = residual
        if residual:
            self.res_connection = nn.Linear(in_feats, out_feats)

        self.bn = batchnorm
        if batchnorm:
            self.bn_layer = nn.BatchNorm1d(out_feats)

    def forward(self, bg, node_feats, etype, norm=None):
        """Update atom representations
        Parameters
        ----------
        bg : BatchedDGLGraph
            Batched DGLGraphs for processing multiple molecules in parallel
        node_feats : FloatTensor of shape (N, M1)
            * N is the total number of atoms in the batched graph
            * M1 is the input atom feature size, must match in_feats in initialization
        etype: int
            bond type
        norm: torch.Tensor
            Optional edge normalizer tensor. Shape: :math:`(|E|, 1)`
        Returns
        -------
        new_feats : FloatTensor of shape (N, M2)
            * M2 is the output atom feature size, must match out_feats in initialization
        """
        new_feats = self.graph_conv_layer(bg, node_feats, etype, norm)
        if self.residual:
            res_feats = self.activation(self.res_connection(node_feats))
            new_feats = new_feats + res_feats
        if self.bn:
            new_feats = self.bn_layer(new_feats)
        del res_feats
        torch.cuda.empty_cache()
        return new_feats


class BaseGNN(nn.Module):
    def __init__(self, gnn_out_feats, n_tasks, rgcn_drop_out=0.5, return_mol_embedding=False, return_weight=False,
                 classifier_hidden_feats=128, dropout=0.):
        super(BaseGNN, self).__init__()
        self.task_num = n_tasks
        self.gnn_layers = nn.ModuleList()
        self.return_weight = return_weight
        self.weighted_sum_readout = WeightAndSum(gnn_out_feats, self.task_num, return_weight=self.return_weight)
        self.fc_in_feats = gnn_out_feats
        self.return_mol_embedding=return_mol_embedding

        self.fc_layers1 = nn.ModuleList([self.fc_layer(dropout, self.fc_in_feats, classifier_hidden_feats) for _ in range(self.task_num)])
        self.fc_layers2 = nn.ModuleList(
            [self.fc_layer(dropout, classifier_hidden_feats, classifier_hidden_feats) for _ in range(self.task_num)])
        self.fc_layers3 = nn.ModuleList(
            [self.fc_layer(dropout, classifier_hidden_feats, classifier_hidden_feats) for _ in range(self.task_num)])

        self.output_layer1 = nn.ModuleList(
            [self.output_layer(classifier_hidden_feats, 1) for _ in range(self.task_num)])

    def forward(self, bg, node_feats, etype, norm=None):
        # Update atom features with GNNs
        for gnn in self.gnn_layers:
            node_feats = gnn(bg, node_feats, etype, norm)

        # Compute molecule features from atom features
        if self.return_weight:
            feats_list, atom_weight_list = self.weighted_sum_readout(bg, node_feats)
        else:
            feats_list = self.weighted_sum_readout(bg, node_feats)

        for i in range(self.task_num):
            # mol_feats = torch.cat([feats_list[-1], feats_list[i]], dim=1)
            mol_feats = feats_list[i]
            h1 = self.fc_layers1[i](mol_feats)
            h2 = self.fc_layers2[i](h1)
            h3 = self.fc_layers3[i](h2)
            predict = self.output_layer1[i](h3)
            if i == 0:
                prediction_all = predict
            else:
                prediction_all = torch.cat([prediction_all, predict], dim=1)
        # generate toxicity fingerprints
        if self.return_mol_embedding:
            return feats_list[0]
        else:
            # generate atom weight and atom feats
            if self.return_weight:
                return prediction_all, atom_weight_list, node_feats
            # just generate prediction
            return prediction_all

    def fc_layer(self, dropout, in_feats, hidden_feats):
        return nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(in_feats, hidden_feats),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_feats)
                )

    def output_layer(self, hidden_feats, out_feats):
        return nn.Sequential(
                nn.Linear(hidden_feats, out_feats)
                )


class MGA(BaseGNN):
    def __init__(self, in_feats, rgcn_hidden_feats, n_tasks, return_weight=False,
                 classifier_hidden_feats=128, loop=False, return_mol_embedding=False,
                 rgcn_drop_out=0.5, dropout=0.):
        super(MGA, self).__init__(gnn_out_feats=rgcn_hidden_feats[-1],
                                  n_tasks=n_tasks,
                                  classifier_hidden_feats=classifier_hidden_feats,
                                  return_mol_embedding=return_mol_embedding,
                                  return_weight=return_weight,
                                  rgcn_drop_out=rgcn_drop_out,
                                  dropout=dropout,
                                  )

        for i in range(len(rgcn_hidden_feats)):
            out_feats = rgcn_hidden_feats[i]
            self.gnn_layers.append(RGCNLayer(in_feats, out_feats, loop=loop, rgcn_drop_out=rgcn_drop_out))
            in_feats = out_feats


def set_random_seed(seed=10):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)


def pos_weight(train_set, classification_num):
    smiles, graphs, labels, mask = map(list, zip(*train_set))
    labels = np.array(labels)
    task_pos_weight_list = []
    for task in range(classification_num):
        num_pos = 0
        num_impos = 0
        for i in labels[:, task]:
            if i == 1:
                num_pos = num_pos + 1
            if i == 0:
                num_impos = num_impos + 1
        weight = num_impos / (num_pos+0.00000001)
        task_pos_weight_list.append(weight)
    task_pos_weight = torch.tensor(task_pos_weight_list)
    return task_pos_weight


class Meter(object):
    """Track and summarize model performance on a dataset for
    (multi-label) binary classification."""
    def __init__(self):
        self.mask = []
        self.y_pred = []
        self.y_true = []

    def update(self, y_pred, y_true, mask):
        """Update for the result of an iteration
        Parameters
        ----------
        y_pred : float32 tensor
            Predicted molecule labels with shape (B, T),
            B for batch size and T for the number of tasks
        y_true : float32 tensor
            Ground truth molecule labels with shape (B, T)
        mask : float32 tensor
            Mask for indicating the existence of ground
            truth labels with shape (B, T)
        """
        self.y_pred.append(y_pred.detach().cpu())
        self.y_true.append(y_true.detach().cpu())
        self.mask.append(mask.detach().cpu())

    def roc_auc_score(self):
        """Compute roc-auc score for each task.
        Returns
        -------
        list of float
            roc-auc score for all tasks
        """
        mask = torch.cat(self.mask, dim=0)
        y_pred = torch.cat(self.y_pred, dim=0)
        y_true = torch.cat(self.y_true, dim=0)
        # Todo: support categorical classes
        # This assumes binary case only
        y_pred = torch.sigmoid(y_pred)
        n_tasks = y_true.shape[1]
        scores = []
        for task in range(n_tasks):
            task_w = mask[:, task]
            task_y_true = y_true[:, task][task_w != 0].numpy()
            task_y_pred = y_pred[:, task][task_w != 0].numpy()
            scores.append(round(roc_auc_score(task_y_true, task_y_pred), 4))
        return scores

    def return_pred_true(self):
        """Compute roc-auc score for each task.
        Returns
        -------
        list of float
            roc-auc score for all tasks
        """
        mask = torch.cat(self.mask, dim=0)
        y_pred = torch.cat(self.y_pred, dim=0)
        y_true = torch.cat(self.y_true, dim=0)
        # Todo: support categorical classes
        # This assumes binary case only
        y_pred = torch.sigmoid(y_pred)
        n_tasks = y_true.shape[1]
        scores = []
        return y_pred, y_true

    def return_pred_true_re(self):
        """Return predicted and true values for regression."""
        y_pred = torch.cat(self.y_pred, dim=0)
        y_true = torch.cat(self.y_true, dim=0)
        return y_pred, y_true

    def l1_loss(self, reduction):
        """Compute l1 loss for each task.
        Returns
        -------
        list of float
            l1 loss for all tasks
        reduction : str
            * 'mean': average the metric over all labeled data points for each task
            * 'sum': sum the metric over all labeled data points for each task
        """
        mask = torch.cat(self.mask, dim=0)
        y_pred = torch.cat(self.y_pred, dim=0)
        y_true = torch.cat(self.y_true, dim=0)
        n_tasks = y_true.shape[1]
        scores = []
        for task in range(n_tasks):
            task_w = mask[:, task]
            task_y_true = y_true[:, task][task_w != 0].numpy()
            task_y_pred = y_pred[:, task][task_w != 0].numpy()
            scores.append(F.l1_loss(task_y_true, task_y_pred, reduction=reduction).item())
        return scores

    def rmse(self):
        """Compute RMSE for each task.
        Returns
        -------
        list of float
            rmse for all tasks
        """
        mask = torch.cat(self.mask, dim=0)
        y_pred = torch.cat(self.y_pred, dim=0)
        y_true = torch.cat(self.y_true, dim=0)
        n_data, n_tasks = y_true.shape
        scores = []
        for task in range(n_tasks):
            task_w = mask[:, task]
            task_y_true = y_true[:, task][task_w != 0].numpy()
            task_y_pred = y_pred[:, task][task_w != 0].numpy()
            scores.append(np.sqrt(F.mse_loss(task_y_pred, task_y_true).cpu().item()))
        return scores

    def mae(self):
        """Compute MAE for each task.
        Returns
        -------
        list of float
            mae for all tasks
        """
        mask = torch.cat(self.mask, dim=0)
        y_pred = torch.cat(self.y_pred, dim=0)
        y_true = torch.cat(self.y_true, dim=0)
        n_data, n_tasks = y_true.shape
        scores = []
        for task in range(n_tasks):
            task_w = mask[:, task]
            task_y_true = y_true[:, task][task_w != 0].numpy()
            task_y_pred = y_pred[:, task][task_w != 0].numpy()
            scores.append(mean_squared_error(task_y_true, task_y_pred))
        return scores

    def r2(self):
        """Compute R2 for each task.
        Returns
        -------
        list of float
            r2 for all tasks
        """
        mask = torch.cat(self.mask, dim=0)
        y_pred = torch.cat(self.y_pred, dim=0)
        y_true = torch.cat(self.y_true, dim=0)
        n_data, n_tasks = y_true.shape
        scores = []
        for task in range(n_tasks):
            task_w = mask[:, task]
            task_y_true = y_true[:, task][task_w != 0].numpy()
            task_y_pred = y_pred[:, task][task_w != 0].numpy()
            scores.append(round(r2_score(task_y_true, task_y_pred), 4))
        return scores

    def roc_precision_recall_score(self):
        """Compute AUC_PRC for each task.
        Returns
        -------
        list of float
            AUC_PRC for all tasks
        """
        mask = torch.cat(self.mask, dim=0)
        y_pred = torch.cat(self.y_pred, dim=0)
        y_true = torch.cat(self.y_true, dim=0)
        # Todo: support categorical classes
        # This assumes binary case only
        y_pred = torch.sigmoid(y_pred)
        n_tasks = y_true.shape[1]
        scores = []
        for task in range(n_tasks):
            task_w = mask[:, task]
            task_y_true = y_true[:, task][task_w != 0].numpy()
            task_y_pred = y_pred[:, task][task_w != 0].numpy()
            precision, recall, _thresholds = precision_recall_curve(task_y_true, task_y_pred)
            scores.append(auc(recall, precision))
        return scores

    def compute_metric(self, metric_name, reduction='mean'):
        """Compute metric for each task.
        Parameters
        ----------
        metric_name : str
            Name for the metric to compute.
        reduction : str
            Only comes into effect when the metric_name is l1_loss.
            * 'mean': average the metric over all labeled data points for each task
            * 'sum': sum the metric over all labeled data points for each task
        Returns
        -------
        list of float
            Metric value for each task
        """
        assert metric_name in ['roc_auc', 'l1', 'rmse', 'mae', 'roc_prc', 'r2', 'return_pred_true'], \
            'Expect metric name to be "roc_auc", "l1" or "rmse", "mae", "roc_prc", "r2", "return_pred_true", got {}'.format(metric_name)
        assert reduction in ['mean', 'sum']
        if metric_name == 'roc_auc':
            return self.roc_auc_score()
        if metric_name == 'l1':
            return self.l1_loss(reduction)
        if metric_name == 'rmse':
            return self.rmse()
        if metric_name == 'mae':
            return self.mae()
        if metric_name == 'roc_prc':
            return self.roc_precision_recall_score()
        if metric_name == 'r2':
            return self.r2()
        if metric_name == 'return_pred_true':
            return self.return_pred_true()


def collate_molgraphs(data):
    smiles, graphs, labels, mask = map(list, zip(*data))
    bg = dgl.batch(graphs)
    bg.set_n_initializer(dgl.init.zero_initializer)
    bg.set_e_initializer(dgl.init.zero_initializer)
    labels = torch.tensor(labels)
    mask = torch.tensor(mask)

    return smiles, bg, labels,  mask


def run_a_train_epoch_heterogeneous(args, epoch, model, data_loader, loss_criterion_c, 
                                    loss_criterion_r, optimizer, task_weight=None):
    model.train()
    train_meter_c = Meter()
    train_meter_r = Meter()
    if task_weight is not None:
        task_weight = task_weight.float().to(args['device'])

    for batch_id, batch_data in enumerate(data_loader):
        smiles, bg, labels, mask = batch_data
        mask = mask.float().to(args['device'])
        labels.float().to(args['device'])
        atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
        bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
        bg = bg.to(args['device'])
        logits = model(bg, atom_feats, bond_feats, norm=None)
        labels = labels.type_as(logits).to(args['device'])
        # calculate loss according to different task class
        if args['task_class'] == 'classification_regression':
            # split classification and regression
            logits_c = logits[:,:args['classification_num']]
            labels_c = labels[:,:args['classification_num']]
            mask_c = mask[:,:args['classification_num']]

            logits_r = logits[:,args['classification_num']:]
            labels_r = labels[:,args['classification_num']:]
            mask_r = mask[:,args['classification_num']:]
            # chose loss function according to task_weight
            if task_weight is None:
                loss = (loss_criterion_c(logits_c, labels_c)*(mask_c != 0).float()).mean() \
                       + (loss_criterion_r(logits_r, labels_r)*(mask_r != 0).float()).mean()
            else:
                task_weight_c = task_weight[:args['classification_num']]
                task_weight_r = task_weight[args['classification_num']:]
                loss = (torch.mean(loss_criterion_c(logits_c, labels_c)*(mask_c != 0).float(), dim=0)*task_weight_c).mean() \
                       + (torch.mean(loss_criterion_r(logits_r, labels_r)*(mask_r != 0).float(), dim=0)*task_weight_r).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            # print('epoch {:d}/{:d}, batch {:d}/{:d}, loss {:.4f}'.format(
            #     epoch + 1, args['num_epochs'], batch_id + 1, len(data_loader), loss.item()))
            train_meter_c.update(logits_c, labels_c, mask_c)
            train_meter_r.update(logits_r, labels_r, mask_r)
            del bg, mask, labels, atom_feats, bond_feats, loss, logits_c, logits_r, labels_c, labels_r, mask_c, mask_r
            torch.cuda.empty_cache()
        elif args['task_class'] == 'classification':
            # chose loss function according to task_weight
            if task_weight is None:
                loss = (loss_criterion_c(logits, labels)*(mask != 0).float()).mean()
            else:
                loss = (torch.mean(loss_criterion_c(logits, labels) * (mask != 0).float(),dim=0)*task_weight).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            # print('epoch {:d}/{:d}, batch {:d}/{:d}, loss {:.4f}'.format(
            #     epoch + 1, args['num_epochs'], batch_id + 1, len(data_loader), loss.item()))
            train_meter_c.update(logits, labels, mask)
            del bg, mask, labels, atom_feats, bond_feats, loss,  logits
            torch.cuda.empty_cache()
        else:
            # chose loss function according to task_weight
            if task_weight is None:
                loss = (loss_criterion_r(logits, labels)*(mask != 0).float()).mean()
            else:
                loss = (torch.mean(loss_criterion_r(logits, labels) * (mask != 0).float(), dim=0)*task_weight).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            # print('epoch {:d}/{:d}, batch {:d}/{:d}, loss {:.4f}'.format(
            #     epoch + 1, args['num_epochs'], batch_id + 1, len(data_loader), loss.item()))
            train_meter_r.update(logits, labels, mask)
            del bg, mask, labels, atom_feats, bond_feats, loss,  logits
            torch.cuda.empty_cache()
    if args['task_class'] == 'classification_regression':
        train_score = np.mean(train_meter_c.compute_metric(args['classification_metric_name']) +
                              train_meter_r.compute_metric(args['regression_metric_name']))
        print('epoch {:d}/{:d}, training {} {:.4f}'.format(
            epoch + 1, args['num_epochs'], 'r2+auc', train_score))
    elif args['task_class'] == 'classification':
        train_score = np.mean(train_meter_c.compute_metric(args['classification_metric_name']))
        print('epoch {:d}/{:d}, training {} {:.4f}'.format(
            epoch + 1, args['num_epochs'], args['classification_metric_name'], train_score))
    else:
        train_score = np.mean(train_meter_r.compute_metric(args['regression_metric_name']))
        print('epoch {:d}/{:d}, training {} {:.4f}'.format(
            epoch + 1, args['num_epochs'], args['regression_metric_name'], train_score))


def run_an_eval_epoch_heterogeneous(args, model, data_loader):
    model.eval()
    eval_meter_c = Meter()
    eval_meter_r = Meter()
    with torch.no_grad():
        for batch_id, batch_data in enumerate(data_loader):
            smiles, bg, labels, mask = batch_data
            labels = labels.float().to(args['device'])
            mask = mask.float().to(args['device'])
            atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
            bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
            bg = bg.to(args['device'])
            logits = model(bg, atom_feats, bond_feats, norm=None)
            labels = labels.type_as(logits).to(args['device'])
            if args['task_class'] == 'classification_regression':
                # split classification and regression
                logits_c = logits[:, :args['classification_num']]
                labels_c = labels[:, :args['classification_num']]
                mask_c = mask[:, :args['classification_num']]
                logits_r = logits[:, args['classification_num']:]
                labels_r = labels[:, args['classification_num']:]
                mask_r = mask[:, args['classification_num']:]
                # Mask non-existing labels
                eval_meter_c.update(logits_c, labels_c, mask_c)
                eval_meter_r.update(logits_r, labels_r, mask_r)
                del smiles, bg,  mask, labels, atom_feats, bond_feats, logits_c, logits_r, labels_c, labels_r, mask_c, mask_r
                torch.cuda.empty_cache()
            elif args['task_class'] == 'classification':
                # Mask non-existing labels
                eval_meter_c.update(logits, labels, mask)
                del smiles, bg,  mask, labels, atom_feats, bond_feats, logits
                torch.cuda.empty_cache()
            else:
                # Mask non-existing labels
                eval_meter_r.update(logits, labels, mask)
                del smiles, bg,  mask, labels, atom_feats, bond_feats, logits
                torch.cuda.empty_cache()
        if args['task_class'] == 'classification_regression':
            return eval_meter_c.compute_metric(args['classification_metric_name']) + \
                   eval_meter_r.compute_metric(args['regression_metric_name'])
        elif args['task_class'] == 'classification':
            return eval_meter_c.compute_metric(args['classification_metric_name'])
        else:
            return eval_meter_r.compute_metric(args['regression_metric_name'])


def run_an_eval_epoch_pih(args, model, data_loader, output_path):
    model.eval()
    eval_meter_c = Meter()
    eval_meter_r = Meter()
    smiles_list = []
    with torch.no_grad():
        for batch_id, batch_data in enumerate(data_loader):
            smiles, bg, labels, mask = batch_data
            smiles_list = smiles_list + smiles
            labels = labels.float().to(args['device'])
            mask = mask.float().to(args['device'])
            atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
            bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
            bg = bg.to(args['device'])
            logits = model(bg, atom_feats, bond_feats, norm=None)
            labels = labels.type_as(logits).to(args['device'])
            if args['task_class'] == 'classification_regression':
                # split classification and regression
                logits_c = logits[:, :args['classification_num']]
                labels_c = labels[:, :args['classification_num']]
                mask_c = mask[:, :args['classification_num']]
                logits_r = logits[:, args['classification_num']:]
                labels_r = labels[:, args['classification_num']:]
                mask_r = mask[:, args['classification_num']:]
                # Mask non-existing labels
                eval_meter_c.update(logits_c, labels_c, mask_c)
                eval_meter_r.update(logits_r, labels_r, mask_r)
                del smiles, bg,  mask, labels, atom_feats, bond_feats, logits_c, logits_r, labels_c, labels_r, mask_c, mask_r
                torch.cuda.empty_cache()
            elif args['task_class'] == 'classification':
                # Mask non-existing labels
                eval_meter_c.update(logits, labels, mask)
                del smiles, bg,  mask, labels, atom_feats, bond_feats, logits
                torch.cuda.empty_cache()
            else:
                # Mask non-existing labels
                eval_meter_r.update(logits, labels, mask)
                del smiles, bg,  mask, labels, atom_feats, bond_feats, logits
                torch.cuda.empty_cache()
        if args['task_class'] == 'classification_regression':
            return eval_meter_c.compute_metric(args['classification_metric_name']) + \
                   eval_meter_r.compute_metric(args['regression_metric_name'])
        elif args['task_class'] == 'classification':
            y_pred, y_true = eval_meter_c.compute_metric('return_pred_true')
            result = pd.DataFrame(columns=['smiles', 'pred', 'true'])
            result['smiles'] = smiles_list
            result['pred'] = np.squeeze(y_pred.numpy()).tolist()
            result['true'] = np.squeeze(y_true.numpy()).tolist()
            result.to_csv(output_path, index=None)
        else:
            # Get predictions and true labels for regression tasks
            y_pred, y_true = eval_meter_r.compute_metric('return_pred_true')
            # Create a DataFrame to store predictions and true labels
            result = pd.DataFrame(columns=['smiles', 'pred', 'true'])
            # Add SMILES information to the DataFrame
            result['smiles'] = smiles_list
            # Add predictions and true labels to the DataFrame
            result['pred'] = np.squeeze(y_pred.numpy()).tolist()
            result['true'] = np.squeeze(y_true.numpy()).tolist()
            # Write the DataFrame to a CSV file
            result.to_csv(output_path, index=None)
            
def run_an_eval_epoch_re(args, model, data_loader, output_path):
    model.eval()
    eval_meter_r = Meter()
    smiles_list = []
    with torch.no_grad():
        for batch_id, batch_data in enumerate(data_loader):
            smiles, bg, labels, mask = batch_data
            smiles_list = smiles_list + smiles
            labels = labels.float().to(args['device'])
            mask = mask.float().to(args['device'])
            atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
            bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
            bg = bg.to(args['device'])
            logits = model(bg, atom_feats, bond_feats, norm=None).to(args['device'])
            labels = labels.type_as(logits).to(args['device'])
            # Mask non-existing labels
            eval_meter_r.update(logits, labels, mask)
            del smiles, bg, mask, labels, atom_feats, bond_feats, logits
            torch.cuda.empty_cache()

        y_pred, y_true = eval_meter_r.return_pred_true_re()

        result = pd.DataFrame(columns=['smiles', 'pred', 'true'])
        result['smiles'] = smiles_list
        result['pred'] = np.squeeze(y_pred.numpy()).tolist()
        result['true'] = np.squeeze(y_true.numpy()).tolist()
        result.to_csv(output_path, index=None)
        
        return y_pred
        


def run_an_eval_epoch_heterogeneous_return_weight(args, model, data_loader, vis_list=None, vis_task='can'):
    model.eval()
    with torch.no_grad():
        for batch_id, batch_data in enumerate(data_loader):
            smiles, bg, labels, mask = batch_data
            #####
            labels = labels.float().to(args['device'])
            atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
            bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
            bg = bg.to(args['device'])
            logits, atom_weight_list, node_feats = model(bg, atom_feats, bond_feats, norm=None)
            labels = labels.type_as(logits).to(args['device'])
            logits_c = logits[:, :args['classification_num']]
            logits_c = torch.sigmoid(logits_c)
            # different tasks with different atom weight

            for mol_index in range(len(smiles)):
                atom_smiles = smiles[mol_index]
                if atom_smiles in vis_list:
                    for tasks_index in range(1):
                        if args['all_task_list'][tasks_index] == vis_task:
                            if labels[mol_index, tasks_index]!=123456:
                                bg.ndata['w'] = atom_weight_list[tasks_index]
                                bg.ndata['feats'] = node_feats
                                unbatch_bg = dgl.unbatch(bg)
                                one_atom_weight = unbatch_bg[mol_index].ndata['w']
                                one_atom_feats = unbatch_bg[mol_index].ndata['feats']
                                # visual selected molecules
                                print('Tasks:', tasks_index, args['all_task_list'][tasks_index], "**********************")
                                if tasks_index < 26:
                                    print('Predict values:', logits[mol_index, tasks_index])
                                else:
                                    print('Predict values:', logits_c[mol_index, tasks_index])
                                print('True values:', labels[mol_index, tasks_index])
                                weight_visulize(atom_smiles, one_atom_weight)
                    else:
                        continue


def run_an_eval_epoch_heterogeneous_return_weight_py(args, model, data_loader, vis_list=None, vis_task='can'):
    model.eval()
    with torch.no_grad():
        for batch_id, batch_data in enumerate(data_loader):
            smiles, bg, labels, mask = batch_data
            #####
            labels = labels.float().to(args['device'])
            atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
            bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
            bg = bg.to(args['device'])
            logits, atom_weight_list, node_feats = model(bg, atom_feats, bond_feats, norm=None)
            labels = labels.type_as(logits).to(args['device'])
            logits_c = logits[:, :args['classification_num']]
            logits_c = torch.sigmoid(logits_c)
            # different tasks with different atom weight

            for mol_index in range(len(smiles)):
                atom_smiles = smiles[mol_index]
                if atom_smiles in vis_list:
                    for tasks_index in range(1):
                        if args['all_task_list'][tasks_index] == vis_task:
                            if labels[mol_index, tasks_index]!=123456:
                                bg.ndata['w'] = atom_weight_list[tasks_index]
                                bg.ndata['feats'] = node_feats
                                unbatch_bg = dgl.unbatch(bg)
                                one_atom_weight = unbatch_bg[mol_index].ndata['w']
                                one_atom_feats = unbatch_bg[mol_index].ndata['feats']
                                # visual selected molecules
                                print('Tasks:', tasks_index, args['all_task_list'][tasks_index], "**********************")
                                if tasks_index < 26:
                                    print('Predict values:', logits_c[mol_index, tasks_index])
                                else:
                                    print('Predict values:', logits[mol_index, tasks_index])
                                print('True values:', labels[mol_index, tasks_index])
                                weight_visulize_py(atom_smiles, one_atom_weight)
                else:
                    continue


def run_an_eval_epoch_heterogeneous_generate_weight(args, model, data_loader):
    model.eval()
    atom_list_all = []
    with torch.no_grad():
        for batch_id, batch_data in enumerate(data_loader):
            print("batch: {}/{}".format(batch_id+1, len(data_loader)))
            smiles, bg, labels, mask = batch_data
            labels = labels.float().to(args['device'])
            atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
            bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
            bg = bg.to(args['device'])
            logits, atom_weight_list = model(bg, atom_feats, bond_feats, norm=None)
            for atom_weight in atom_weight_list:
                atom_list_all.append(atom_weight[args['select_task_index']])
    task_name = args['select_task_list'][0]
    atom_weight_list = pd.DataFrame(atom_list_all, columns=['atom_weight'])
    atom_weight_list.to_csv(task_name+"_atom_weight.csv", index=None)


def generate_chemical_environment(args, model, data_loader):
    model.eval()
    atom_list_all = []
    with torch.no_grad():
        for batch_id, batch_data in enumerate(data_loader):
            print("batch: {}/{}".format(batch_id + 1, len(data_loader)))
            smiles, bg, labels, mask = batch_data
            print(bg.ndata[args['atom_data_field']][1])
            atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
            bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
            bg = bg.to(args['device'])
            logits, atom_weight_list = model(bg, atom_feats, bond_feats, norm=None)
            print('after training:', bg.ndata['h'][1])


def generate_mol_feats(args, model, data_loader, dataset_output_path):
    model.eval()
    with torch.no_grad():
        for batch_id, batch_data in enumerate(data_loader):
            smiles, bg, labels, mask = batch_data
            bg = bg.to(args['device'])
            atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
            bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
            feats = model(bg, atom_feats, bond_feats, norm=None).numpy().tolist()
            feats_name = ['graph-feature' + str(i+1) for i in range(64)]
            data = pd.DataFrame(feats, columns=feats_name)
            data['smiles'] = smiles
            data['labels'] = labels.squeeze().numpy().tolist()
    data.to_csv(dataset_output_path, index=None)


class EarlyStopping(object):
    def __init__(self, pretrained_model='Null_early_stop.pth', mode='higher', patience=10, filename=None, task_name="None"):
        if filename is None:
            task_name = task_name
            filename ='../model/{}_early_stop.pth'.format(task_name)

        assert mode in ['higher', 'lower']
        self.mode = mode
        if self.mode == 'higher':
            self._check = self._check_higher
        else:
            self._check = self._check_lower

        self.patience = patience
        self.counter = 0
        self.filename = filename
        self.best_score = None
        self.early_stop = False
        self.pretrained_model = pretrained_model

    def _check_higher(self, score, prev_best_score):
        return (score > prev_best_score)

    def _check_lower(self, score, prev_best_score):
        return (score < prev_best_score)

    def step(self, score, model):
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(model)
        elif self._check(score, self.best_score):
            self.best_score = score
            self.save_checkpoint(model)
            self.counter = 0
        else:
            self.counter += 1
            print(
                'EarlyStopping counter: {} out of {}'.format(self.counter, self.patience))
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

    def nosave_step(self, score):
        if self.best_score is None:
            self.best_score = score
        elif self._check(score, self.best_score):
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            print(
                'EarlyStopping counter: {} out of {}'.format(self.counter, self.patience))
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

    def save_checkpoint(self, model):
        '''Saves model when the metric on the validation set gets improved.'''
        torch.save({'model_state_dict': model.state_dict()}, self.filename)
        # print(self.filename)

    def load_checkpoint(self, model):
        '''Load model saved with early stopping.'''
        # model.load_state_dict(torch.load(self.filename)['model_state_dict'])
        model.load_state_dict(torch.load(self.filename, map_location=torch.device('cpu'))['model_state_dict'])

    def load_pretrained_model(self, model):
        pretrained_parameters = ['gnn_layers.0.graph_conv_layer.weight',
                                 'gnn_layers.0.graph_conv_layer.h_bias',
                                 'gnn_layers.0.graph_conv_layer.loop_weight',
                                 'gnn_layers.0.res_connection.weight',
                                 'gnn_layers.0.res_connection.bias',
                                 'gnn_layers.0.bn_layer.weight',
                                 'gnn_layers.0.bn_layer.bias',
                                 'gnn_layers.0.bn_layer.running_mean',
                                 'gnn_layers.0.bn_layer.running_var',
                                 'gnn_layers.0.bn_layer.num_batches_tracked',
                                 'gnn_layers.1.graph_conv_layer.weight',
                                 'gnn_layers.1.graph_conv_layer.h_bias',
                                 'gnn_layers.1.graph_conv_layer.loop_weight',
                                 'gnn_layers.1.res_connection.weight',
                                 'gnn_layers.1.res_connection.bias',
                                 'gnn_layers.1.bn_layer.weight',
                                 'gnn_layers.1.bn_layer.bias',
                                 'gnn_layers.1.bn_layer.running_mean',
                                 'gnn_layers.1.bn_layer.running_var',
                                 'gnn_layers.1.bn_layer.num_batches_tracked']
        if torch.cuda.is_available():
            pretrained_model = torch.load('../model/'+self.pretrained_model)
        else:
            pretrained_model = torch.load('../model/'+self.pretrained_model, map_location=torch.device('cpu'))
        model_dict = model.state_dict()
        pretrained_dict = {k: v for k, v in pretrained_model['model_state_dict'].items() if k in pretrained_parameters}
        model_dict.update(pretrained_dict)
        model.load_state_dict(pretrained_dict, strict=False)

    def load_model_attention(self, model):
        pretrained_parameters = ['gnn_layers.0.graph_conv_layer.weight',
                                 'gnn_layers.0.graph_conv_layer.h_bias',
                                 'gnn_layers.0.graph_conv_layer.loop_weight',
                                 'gnn_layers.0.res_connection.weight',
                                 'gnn_layers.0.res_connection.bias',
                                 'gnn_layers.0.bn_layer.weight',
                                 'gnn_layers.0.bn_layer.bias',
                                 'gnn_layers.0.bn_layer.running_mean',
                                 'gnn_layers.0.bn_layer.running_var',
                                 'gnn_layers.0.bn_layer.num_batches_tracked',
                                 'gnn_layers.1.graph_conv_layer.weight',
                                 'gnn_layers.1.graph_conv_layer.h_bias',
                                 'gnn_layers.1.graph_conv_layer.loop_weight',
                                 'gnn_layers.1.res_connection.weight',
                                 'gnn_layers.1.res_connection.bias',
                                 'gnn_layers.1.bn_layer.weight',
                                 'gnn_layers.1.bn_layer.bias',
                                 'gnn_layers.1.bn_layer.running_mean',
                                 'gnn_layers.1.bn_layer.running_var',
                                 'gnn_layers.1.bn_layer.num_batches_tracked',
                                 'weighted_sum_readout.atom_weighting_specific.0.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.0.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.1.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.1.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.2.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.2.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.3.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.3.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.4.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.4.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.5.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.5.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.6.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.6.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.7.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.7.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.8.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.8.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.9.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.9.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.10.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.10.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.11.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.11.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.12.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.12.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.13.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.13.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.14.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.14.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.15.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.15.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.16.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.16.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.17.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.17.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.18.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.18.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.19.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.19.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.20.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.20.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.21.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.21.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.22.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.22.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.23.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.23.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.24.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.24.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.25.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.25.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.26.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.26.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.27.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.27.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.28.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.28.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.29.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.29.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.30.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.30.0.bias',
                                 'weighted_sum_readout.shared_weighting.0.weight',
                                 'weighted_sum_readout.shared_weighting.0.bias',
                                 ]
        if torch.cuda.is_available():
            pretrained_model = torch.load('../model/' + self.pretrained_model)
        else:
            pretrained_model = torch.load('../model/' + self.pretrained_model, map_location=torch.device('cpu'))
        model_dict = model.state_dict()
        pretrained_dict = {k: v for k, v in pretrained_model['model_state_dict'].items() if k in pretrained_parameters}
        model_dict.update(pretrained_dict)
        model.load_state_dict(pretrained_dict, strict=False)

In [ ]:
def run_a_train_epoch_heterogeneous(args, epoch, model, data_loader, loss_criterion_c, loss_criterion_r, optimizer, task_weight,regression_indices):
    model.train()
    train_meter_c = Meter()
    train_meter_r = Meter()
    if task_weight is not None:
        task_weight = task_weight.float().to(args['device'])

    # Determine the indices of regression tasks
    regression_task_names = ['ISF', 'IUR', 'OSF']
    regression_indices = [args['select_task_list'].index(task) - args['classification_num'] 
                         for task in regression_task_names if task in args['select_task_list']]

    for batch_id, batch_data in enumerate(data_loader):
        smiles, bg, labels, mask = batch_data
        mask = mask.float().to(args['device'])
        labels.float().to(args['device'])
        atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
        bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
        bg = bg.to(args['device'])
        logits = model(bg, atom_feats, bond_feats, norm=None)
        labels = labels.type_as(logits).to(args['device'])
        
        # calculate loss according to different task class
        if args['task_class'] == 'classification_regression':
            # split classification and regression
            logits_c = logits[:,:args['classification_num']]
            labels_c = labels[:,:args['classification_num']]
            mask_c = mask[:,:args['classification_num']]

            logits_r = logits[:,args['classification_num']:]
            labels_r = labels[:,args['classification_num']:]
            mask_r = mask[:,args['classification_num']:]
            
            # Compute classification loss
            class_loss = (loss_criterion_c(logits_c, labels_c)*(mask_c != 0).float()).mean()
            
            # Compute regression loss using dynamic weights
            if task_weight is not None and len(regression_indices) > 0:
                reg_losses = []
                for i, reg_idx in enumerate(regression_indices):
                    # Ensure indices are within the valid range
                    if reg_idx < logits_r.shape[1]:
                        task_mask = mask_r[:, reg_idx:reg_idx+1]
                        task_logits = logits_r[:, reg_idx:reg_idx+1]
                        task_labels = labels_r[:, reg_idx:reg_idx+1]
                        
                        # Compute the loss for each individual task
                        task_loss = loss_criterion_r(task_logits, task_labels) * (task_mask != 0).float()
                        
                        # If samples are available for this task, compute the weighted loss
                        if torch.sum(task_mask) > 0:
                            task_loss = torch.sum(task_loss) / torch.sum(task_mask)
                            reg_losses.append(task_weight[i] * task_loss)
                
                # If valid regression tasks exist, compute the average loss
                if len(reg_losses) > 0:
                    reg_loss = sum(reg_losses) / len(reg_losses)
                else:
                    reg_loss = 0
            else:
                # Compute regression loss using the original method
                reg_loss = (loss_criterion_r(logits_r, labels_r)*(mask_r != 0).float()).mean()
            
            # Total loss
            loss = class_loss + reg_loss
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # print('epoch {:d}/{:d}, batch {:d}/{:d}, loss {:.4f}'.format(
            #     epoch + 1, args['num_epochs'], batch_id + 1, len(data_loader), loss.item()))
            train_meter_c.update(logits_c, labels_c, mask_c)
            train_meter_r.update(logits_r, labels_r, mask_r)
            del bg, mask, labels, atom_feats, bond_feats, loss, logits_c, logits_r, labels_c, labels_r, mask_c, mask_r
            torch.cuda.empty_cache()
        elif args['task_class'] == 'classification':
            # chose loss function according to task_weight
            if task_weight is None:
                loss = (loss_criterion_c(logits, labels)*(mask != 0).float()).mean()
            else:
                loss = (torch.mean(loss_criterion_c(logits, labels) * (mask != 0).float(),dim=0)*task_weight).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            # print('epoch {:d}/{:d}, batch {:d}/{:d}, loss {:.4f}'.format(
            #     epoch + 1, args['num_epochs'], batch_id + 1, len(data_loader), loss.item()))
            train_meter_c.update(logits, labels, mask)
            del bg, mask, labels, atom_feats, bond_feats, loss,  logits
            torch.cuda.empty_cache()
        else:  # Regression-only tasks
            # Compute regression loss using dynamic weights
            if task_weight is not None and len(regression_indices) > 0:
                reg_losses = []
                for i, reg_idx in enumerate(regression_indices):
                    if reg_idx < logits.shape[1]:
                        task_mask = mask[:, reg_idx:reg_idx+1]
                        task_logits = logits[:, reg_idx:reg_idx+1]
                        task_labels = labels[:, reg_idx:reg_idx+1]
                        
                        task_loss = loss_criterion_r(task_logits, task_labels) * (task_mask != 0).float()
                        
                        if torch.sum(task_mask) > 0:
                            task_loss = torch.sum(task_loss) / torch.sum(task_mask)
                            reg_losses.append(task_weight[i] * task_loss)
                
                if len(reg_losses) > 0:
                    loss = sum(reg_losses) / len(reg_losses)
                else:
                    loss = (loss_criterion_r(logits, labels)*(mask != 0).float()).mean()
            else:
                loss = (loss_criterion_r(logits, labels)*(mask != 0).float()).mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            # print('epoch {:d}/{:d}, batch {:d}/{:d}, loss {:.4f}'.format(
            #     epoch + 1, args['num_epochs'], batch_id + 1, len(data_loader), loss.item()))
            train_meter_r.update(logits, labels, mask)
            del bg, mask, labels, atom_feats, bond_feats, loss,  logits
            torch.cuda.empty_cache()
    
    if args['task_class'] == 'classification_regression':
        train_score = np.mean(train_meter_c.compute_metric(args['classification_metric_name']) +
                              train_meter_r.compute_metric(args['regression_metric_name']))
        print('epoch {:d}/{:d}, training {} {:.4f}'.format(
            epoch + 1, args['num_epochs'], 'r2+auc', train_score))
    elif args['task_class'] == 'classification':
        train_score = np.mean(train_meter_c.compute_metric(args['classification_metric_name']))
        print('epoch {:d}/{:d}, training {} {:.4f}'.format(
            epoch + 1, args['num_epochs'], args['classification_metric_name'], train_score))
    else:
        train_score = np.mean(train_meter_r.compute_metric(args['regression_metric_name']))
        print('epoch {:d}/{:d}, training {} {:.4f}'.format(
            epoch + 1, args['num_epochs'], args['regression_metric_name'], train_score))
        
        def run_a_train_epoch_heterogeneous(args, epoch, model, data_loader, loss_criterion_c, loss_criterion_r, optimizer, task_weight):
            model.train()
            train_meter_c = Meter()
            train_meter_r = Meter()
    if task_weight is not None:
        task_weight = task_weight.float().to(args['device'])

    # Determine the indices of regression tasks
    regression_task_names = ['ISF', 'IUR', 'OSF']
    regression_indices = [args['select_task_list'].index(task) - args['classification_num'] 
                         for task in regression_task_names if task in args['select_task_list']]

    for batch_id, batch_data in enumerate(data_loader):
        smiles, bg, labels, mask = batch_data
        mask = mask.float().to(args['device'])
        labels.float().to(args['device'])
        atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
        bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
        bg = bg.to(args['device'])
        logits = model(bg, atom_feats, bond_feats, norm=None)
        labels = labels.type_as(logits).to(args['device'])
        
        # calculate loss according to different task class
        if args['task_class'] == 'classification_regression':
            # split classification and regression
            logits_c = logits[:,:args['classification_num']]
            labels_c = labels[:,:args['classification_num']]
            mask_c = mask[:,:args['classification_num']]

            logits_r = logits[:,args['classification_num']:]
            labels_r = labels[:,args['classification_num']:]
            mask_r = mask[:,args['classification_num']:]
            
            # Compute classification loss
            class_loss = (loss_criterion_c(logits_c, labels_c)*(mask_c != 0).float()).mean()
            
            # Compute regression loss using dynamic weights
            if task_weight is not None and len(regression_indices) > 0:
                reg_losses = []
                for i, reg_idx in enumerate(regression_indices):
                    # Ensure indices are within the valid range
                    if reg_idx < logits_r.shape[1]:
                        task_mask = mask_r[:, reg_idx:reg_idx+1]
                        task_logits = logits_r[:, reg_idx:reg_idx+1]
                        task_labels = labels_r[:, reg_idx:reg_idx+1]
                        
                        # Compute the loss for each individual task
                        task_loss = loss_criterion_r(task_logits, task_labels) * (task_mask != 0).float()
                        
                        # If samples are available for this task, compute the weighted loss
                        if torch.sum(task_mask) > 0:
                            task_loss = torch.sum(task_loss) / torch.sum(task_mask)
                            reg_losses.append(task_weight[i] * task_loss)
                
                # If valid regression tasks exist, compute the average loss
                if len(reg_losses) > 0:
                    reg_loss = sum(reg_losses) / len(reg_losses)
                else:
                    reg_loss = 0
            else:
                # Compute regression loss using the original method
                reg_loss = (loss_criterion_r(logits_r, labels_r)*(mask_r != 0).float()).mean()
            
            # Total loss
            loss = class_loss + reg_loss
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # print('epoch {:d}/{:d}, batch {:d}/{:d}, loss {:.4f}'.format(
            #     epoch + 1, args['num_epochs'], batch_id + 1, len(data_loader), loss.item()))
            train_meter_c.update(logits_c, labels_c, mask_c)
            train_meter_r.update(logits_r, labels_r, mask_r)
            del bg, mask, labels, atom_feats, bond_feats, loss, logits_c, logits_r, labels_c, labels_r, mask_c, mask_r
            torch.cuda.empty_cache()
        elif args['task_class'] == 'classification':
            # chose loss function according to task_weight
            if task_weight is None:
                loss = (loss_criterion_c(logits, labels)*(mask != 0).float()).mean()
            else:
                loss = (torch.mean(loss_criterion_c(logits, labels) * (mask != 0).float(),dim=0)*task_weight).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            # print('epoch {:d}/{:d}, batch {:d}/{:d}, loss {:.4f}'.format(
            #     epoch + 1, args['num_epochs'], batch_id + 1, len(data_loader), loss.item()))
            train_meter_c.update(logits, labels, mask)
            del bg, mask, labels, atom_feats, bond_feats, loss,  logits
            torch.cuda.empty_cache()
        else:  # Regression-only tasks
            # Compute regression loss using dynamic weights
            if task_weight is not None and len(regression_indices) > 0:
                reg_losses = []
                for i, reg_idx in enumerate(regression_indices):
                    if reg_idx < logits.shape[1]:
                        task_mask = mask[:, reg_idx:reg_idx+1]
                        task_logits = logits[:, reg_idx:reg_idx+1]
                        task_labels = labels[:, reg_idx:reg_idx+1]
                        
                        task_loss = loss_criterion_r(task_logits, task_labels) * (task_mask != 0).float()
                        
                        if torch.sum(task_mask) > 0:
                            task_loss = torch.sum(task_loss) / torch.sum(task_mask)
                            reg_losses.append(task_weight[i] * task_loss)
                
                if len(reg_losses) > 0:
                    loss = sum(reg_losses) / len(reg_losses)
                else:
                    loss = (loss_criterion_r(logits, labels)*(mask != 0).float()).mean()
            else:
                loss = (loss_criterion_r(logits, labels)*(mask != 0).float()).mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            # print('epoch {:d}/{:d}, batch {:d}/{:d}, loss {:.4f}'.format(
            #     epoch + 1, args['num_epochs'], batch_id + 1, len(data_loader), loss.item()))
            train_meter_r.update(logits, labels, mask)
            del bg, mask, labels, atom_feats, bond_feats, loss,  logits
            torch.cuda.empty_cache()
    
    if args['task_class'] == 'classification_regression':
        train_score = np.mean(train_meter_c.compute_metric(args['classification_metric_name']) +
                              train_meter_r.compute_metric(args['regression_metric_name']))
        print('epoch {:d}/{:d}, training {} {:.4f}'.format(
            epoch + 1, args['num_epochs'], 'r2+auc', train_score))
    elif args['task_class'] == 'classification':
        train_score = np.mean(train_meter_c.compute_metric(args['classification_metric_name']))
        print('epoch {:d}/{:d}, training {} {:.4f}'.format(
            epoch + 1, args['num_epochs'], args['classification_metric_name'], train_score))
    else:
        train_score = np.mean(train_meter_r.compute_metric(args['regression_metric_name']))
        print('epoch {:d}/{:d}, training {} {:.4f}'.format(
            epoch + 1, args['num_epochs'], args['regression_metric_name'], train_score))

In [ ]:
import datetime
from sklearn.metrics import roc_auc_score, mean_squared_error, precision_recall_curve, auc, r2_score
import torch
import torch.nn.functional as F
import dgl
import numpy as np
import random
from dgl.readout import sum_nodes
from dgl.nn.pytorch.conv import RelGraphConv
from torch import nn
import pandas as pd



class WeightAndSum(nn.Module):
    def __init__(self, in_feats, task_num=1, attention=True, return_weight=False):
        super(WeightAndSum, self).__init__()
        self.attention = attention
        self.in_feats = in_feats
        self.task_num = task_num
        self.return_weight=return_weight
        self.atom_weighting_specific = nn.ModuleList([self.atom_weight(self.in_feats) for _ in range(self.task_num)])
        self.shared_weighting = self.atom_weight(self.in_feats)
    def forward(self, bg, feats):
        feat_list = []
        atom_list = []
        # cal specific feats
        for i in range(self.task_num):
            with bg.local_scope():
                bg.ndata['h'] = feats
                weight = self.atom_weighting_specific[i](feats)
                bg.ndata['w'] = weight
                specific_feats_sum = sum_nodes(bg, 'h', 'w')
                atom_list.append(bg.ndata['w'])
            feat_list.append(specific_feats_sum)

        # cal shared feats
        with bg.local_scope():
            bg.ndata['h'] = feats
            bg.ndata['w'] = self.shared_weighting(feats)
            shared_feats_sum = sum_nodes(bg, 'h', 'w')
        # feat_list.append(shared_feats_sum)
        if self.attention:
            if self.return_weight:
                return feat_list, atom_list
            else:
                return feat_list
        else:
            return shared_feats_sum

    def atom_weight(self, in_feats):
        return nn.Sequential(
            nn.Linear(in_feats, 1),
            nn.Sigmoid()
            )


class MLPBinaryClassifier(nn.Module):
    def __init__(self, in_feats, hidden_feats, n_tasks, dropout=0.):
        super(MLPBinaryClassifier, self).__init__()

        self.predict = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_feats, hidden_feats),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_feats),
            nn.Linear(hidden_feats, n_tasks)
        )

    def forward(self, h):
        return self.predict(h)


class RGCNLayer(nn.Module):
    def __init__(self, in_feats, out_feats, num_rels=64*21, activation=F.relu, loop=False,
                 residual=True, batchnorm=True, rgcn_drop_out=0.5):
        super(RGCNLayer, self).__init__()

        self.activation = activation
        self.graph_conv_layer = RelGraphConv(in_feats, out_feats, num_rels=num_rels, regularizer='basis',
                                               num_bases=None, bias=True, activation=activation,
                                               self_loop=loop, dropout=rgcn_drop_out)
        self.residual = residual
        if residual:
            self.res_connection = nn.Linear(in_feats, out_feats)

        self.bn = batchnorm
        if batchnorm:
            self.bn_layer = nn.BatchNorm1d(out_feats)

    def forward(self, bg, node_feats, etype, norm=None):
        """Update atom representations
        Parameters
        ----------
        bg : BatchedDGLGraph
            Batched DGLGraphs for processing multiple molecules in parallel
        node_feats : FloatTensor of shape (N, M1)
            * N is the total number of atoms in the batched graph
            * M1 is the input atom feature size, must match in_feats in initialization
        etype: int
            bond type
        norm: torch.Tensor
            Optional edge normalizer tensor. Shape: :math:`(|E|, 1)`
        Returns
        -------
        new_feats : FloatTensor of shape (N, M2)
            * M2 is the output atom feature size, must match out_feats in initialization
        """
        new_feats = self.graph_conv_layer(bg, node_feats, etype, norm)
        if self.residual:
            res_feats = self.activation(self.res_connection(node_feats))
            new_feats = new_feats + res_feats
        if self.bn:
            new_feats = self.bn_layer(new_feats)
        del res_feats
        torch.cuda.empty_cache()
        return new_feats


class BaseGNN(nn.Module):
    def __init__(self, gnn_out_feats, n_tasks, rgcn_drop_out=0.5, return_mol_embedding=False, return_weight=False,
                 classifier_hidden_feats=128, dropout=0.):
        super(BaseGNN, self).__init__()
        self.task_num = n_tasks
        self.gnn_layers = nn.ModuleList()
        self.return_weight = return_weight
        self.weighted_sum_readout = WeightAndSum(gnn_out_feats, self.task_num, return_weight=self.return_weight)
        self.fc_in_feats = gnn_out_feats
        self.return_mol_embedding=return_mol_embedding

        self.fc_layers1 = nn.ModuleList([self.fc_layer(dropout, self.fc_in_feats, classifier_hidden_feats) for _ in range(self.task_num)])
        self.fc_layers2 = nn.ModuleList(
            [self.fc_layer(dropout, classifier_hidden_feats, classifier_hidden_feats) for _ in range(self.task_num)])
        self.fc_layers3 = nn.ModuleList(
            [self.fc_layer(dropout, classifier_hidden_feats, classifier_hidden_feats) for _ in range(self.task_num)])

        self.output_layer1 = nn.ModuleList(
            [self.output_layer(classifier_hidden_feats, 1) for _ in range(self.task_num)])

    def forward(self, bg, node_feats, etype, norm=None):
        # Update atom features with GNNs
        for gnn in self.gnn_layers:
            node_feats = gnn(bg, node_feats, etype, norm)

        # Compute molecule features from atom features
        if self.return_weight:
            feats_list, atom_weight_list = self.weighted_sum_readout(bg, node_feats)
        else:
            feats_list = self.weighted_sum_readout(bg, node_feats)

        for i in range(self.task_num):
            # mol_feats = torch.cat([feats_list[-1], feats_list[i]], dim=1)
            mol_feats = feats_list[i]
            h1 = self.fc_layers1[i](mol_feats)
            h2 = self.fc_layers2[i](h1)
            h3 = self.fc_layers3[i](h2)
            predict = self.output_layer1[i](h3)
            if i == 0:
                prediction_all = predict
            else:
                prediction_all = torch.cat([prediction_all, predict], dim=1)
        # generate toxicity fingerprints
        if self.return_mol_embedding:
            return feats_list[0]
        else:
            # generate atom weight and atom feats
            if self.return_weight:
                return prediction_all, atom_weight_list, node_feats
            # just generate prediction
            return prediction_all

    def fc_layer(self, dropout, in_feats, hidden_feats):
        return nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(in_feats, hidden_feats),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_feats)
                )

    def output_layer(self, hidden_feats, out_feats):
        return nn.Sequential(
                nn.Linear(hidden_feats, out_feats)
                )


class MGA(BaseGNN):
    def __init__(self, in_feats, rgcn_hidden_feats, n_tasks, return_weight=False,
                 classifier_hidden_feats=128, loop=False, return_mol_embedding=False,
                 rgcn_drop_out=0.5, dropout=0.):
        super(MGA, self).__init__(gnn_out_feats=rgcn_hidden_feats[-1],
                                  n_tasks=n_tasks,
                                  classifier_hidden_feats=classifier_hidden_feats,
                                  return_mol_embedding=return_mol_embedding,
                                  return_weight=return_weight,
                                  rgcn_drop_out=rgcn_drop_out,
                                  dropout=dropout,
                                  )

        for i in range(len(rgcn_hidden_feats)):
            out_feats = rgcn_hidden_feats[i]
            self.gnn_layers.append(RGCNLayer(in_feats, out_feats, loop=loop, rgcn_drop_out=rgcn_drop_out))
            in_feats = out_feats


def set_random_seed(seed=10):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)


def pos_weight(train_set, classification_num):
    smiles, graphs, labels, mask = map(list, zip(*train_set))
    labels = np.array(labels)
    task_pos_weight_list = []
    for task in range(classification_num):
        num_pos = 0
        num_impos = 0
        for i in labels[:, task]:
            if i == 1:
                num_pos = num_pos + 1
            if i == 0:
                num_impos = num_impos + 1
        weight = num_impos / (num_pos+0.00000001)
        task_pos_weight_list.append(weight)
    task_pos_weight = torch.tensor(task_pos_weight_list)
    return task_pos_weight


class Meter(object):
    """Track and summarize model performance on a dataset for
    (multi-label) binary classification."""
    def __init__(self):
        self.mask = []
        self.y_pred = []
        self.y_true = []

    def update(self, y_pred, y_true, mask):
        """Update for the result of an iteration
        Parameters
        ----------
        y_pred : float32 tensor
            Predicted molecule labels with shape (B, T),
            B for batch size and T for the number of tasks
        y_true : float32 tensor
            Ground truth molecule labels with shape (B, T)
        mask : float32 tensor
            Mask for indicating the existence of ground
            truth labels with shape (B, T)
        """
        self.y_pred.append(y_pred.detach().cpu())
        self.y_true.append(y_true.detach().cpu())
        self.mask.append(mask.detach().cpu())

    def roc_auc_score(self):
        """Compute roc-auc score for each task.
        Returns
        -------
        list of float
            roc-auc score for all tasks
        """
        mask = torch.cat(self.mask, dim=0)
        y_pred = torch.cat(self.y_pred, dim=0)
        y_true = torch.cat(self.y_true, dim=0)
        # Todo: support categorical classes
        # This assumes binary case only
        y_pred = torch.sigmoid(y_pred)
        n_tasks = y_true.shape[1]
        scores = []
        for task in range(n_tasks):
            task_w = mask[:, task]
            task_y_true = y_true[:, task][task_w != 0].numpy()
            task_y_pred = y_pred[:, task][task_w != 0].numpy()
            scores.append(round(roc_auc_score(task_y_true, task_y_pred), 4))
        return scores

    def return_pred_true(self):
        """Compute roc-auc score for each task.
        Returns
        -------
        list of float
            roc-auc score for all tasks
        """
        mask = torch.cat(self.mask, dim=0)
        y_pred = torch.cat(self.y_pred, dim=0)
        y_true = torch.cat(self.y_true, dim=0)
        # Todo: support categorical classes
        # This assumes binary case only
        y_pred = torch.sigmoid(y_pred)
        n_tasks = y_true.shape[1]
        scores = []
        return y_pred, y_true

    def return_pred_true_re(self):
        """Return predicted and true values for regression."""
        y_pred = torch.cat(self.y_pred, dim=0)
        y_true = torch.cat(self.y_true, dim=0)
        return y_pred, y_true

    def l1_loss(self, reduction):
        """Compute l1 loss for each task.
        Returns
        -------
        list of float
            l1 loss for all tasks
        reduction : str
            * 'mean': average the metric over all labeled data points for each task
            * 'sum': sum the metric over all labeled data points for each task
        """
        mask = torch.cat(self.mask, dim=0)
        y_pred = torch.cat(self.y_pred, dim=0)
        y_true = torch.cat(self.y_true, dim=0)
        n_tasks = y_true.shape[1]
        scores = []
        for task in range(n_tasks):
            task_w = mask[:, task]
            task_y_true = y_true[:, task][task_w != 0].numpy()
            task_y_pred = y_pred[:, task][task_w != 0].numpy()
            scores.append(F.l1_loss(task_y_true, task_y_pred, reduction=reduction).item())
        return scores

    def rmse(self):
        """Compute RMSE for each task.
        Returns
        -------
        list of float
            rmse for all tasks
        """
        mask = torch.cat(self.mask, dim=0)
        y_pred = torch.cat(self.y_pred, dim=0)
        y_true = torch.cat(self.y_true, dim=0)
        n_data, n_tasks = y_true.shape
        scores = []
        for task in range(n_tasks):
            task_w = mask[:, task]
            task_y_true = y_true[:, task][task_w != 0].numpy()
            task_y_pred = y_pred[:, task][task_w != 0].numpy()
            scores.append(np.sqrt(F.mse_loss(task_y_pred, task_y_true).cpu().item()))
        return scores

    def mae(self):
        """Compute MAE for each task.
        Returns
        -------
        list of float
            mae for all tasks
        """
        mask = torch.cat(self.mask, dim=0)
        y_pred = torch.cat(self.y_pred, dim=0)
        y_true = torch.cat(self.y_true, dim=0)
        n_data, n_tasks = y_true.shape
        scores = []
        for task in range(n_tasks):
            task_w = mask[:, task]
            task_y_true = y_true[:, task][task_w != 0].numpy()
            task_y_pred = y_pred[:, task][task_w != 0].numpy()
            scores.append(mean_squared_error(task_y_true, task_y_pred))
        return scores

    def r2(self):
        """Compute R2 for each task.
        Returns
        -------
        list of float
            r2 for all tasks
        """
        mask = torch.cat(self.mask, dim=0)
        y_pred = torch.cat(self.y_pred, dim=0)
        y_true = torch.cat(self.y_true, dim=0)
        n_data, n_tasks = y_true.shape
        scores = []
        for task in range(n_tasks):
            task_w = mask[:, task]
            task_y_true = y_true[:, task][task_w != 0].numpy()
            task_y_pred = y_pred[:, task][task_w != 0].numpy()
            scores.append(round(r2_score(task_y_true, task_y_pred), 4))
        return scores

    def roc_precision_recall_score(self):
        """Compute AUC_PRC for each task.
        Returns
        -------
        list of float
            AUC_PRC for all tasks
        """
        mask = torch.cat(self.mask, dim=0)
        y_pred = torch.cat(self.y_pred, dim=0)
        y_true = torch.cat(self.y_true, dim=0)
        # Todo: support categorical classes
        # This assumes binary case only
        y_pred = torch.sigmoid(y_pred)
        n_tasks = y_true.shape[1]
        scores = []
        for task in range(n_tasks):
            task_w = mask[:, task]
            task_y_true = y_true[:, task][task_w != 0].numpy()
            task_y_pred = y_pred[:, task][task_w != 0].numpy()
            precision, recall, _thresholds = precision_recall_curve(task_y_true, task_y_pred)
            scores.append(auc(recall, precision))
        return scores

    def compute_metric(self, metric_name, reduction='mean'):
        """Compute metric for each task.
        Parameters
        ----------
        metric_name : str
            Name for the metric to compute.
        reduction : str
            Only comes into effect when the metric_name is l1_loss.
            * 'mean': average the metric over all labeled data points for each task
            * 'sum': sum the metric over all labeled data points for each task
        Returns
        -------
        list of float
            Metric value for each task
        """
        assert metric_name in ['roc_auc', 'l1', 'rmse', 'mae', 'roc_prc', 'r2', 'return_pred_true'], \
            'Expect metric name to be "roc_auc", "l1" or "rmse", "mae", "roc_prc", "r2", "return_pred_true", got {}'.format(metric_name)
        assert reduction in ['mean', 'sum']
        if metric_name == 'roc_auc':
            return self.roc_auc_score()
        if metric_name == 'l1':
            return self.l1_loss(reduction)
        if metric_name == 'rmse':
            return self.rmse()
        if metric_name == 'mae':
            return self.mae()
        if metric_name == 'roc_prc':
            return self.roc_precision_recall_score()
        if metric_name == 'r2':
            return self.r2()
        if metric_name == 'return_pred_true':
            return self.return_pred_true()


def collate_molgraphs(data):
    smiles, graphs, labels, mask = map(list, zip(*data))
    bg = dgl.batch(graphs)
    bg.set_n_initializer(dgl.init.zero_initializer)
    bg.set_e_initializer(dgl.init.zero_initializer)
    labels = torch.tensor(labels)
    mask = torch.tensor(mask)

    return smiles, bg, labels,  mask

def run_a_train_epoch_heterogeneous(
    args, epoch, model, data_loader, 
    loss_criterion_c, loss_criterion_r, 
    optimizer, 
    reg_prior_weights=None  # Added: prior weights for regression tasks (user-defined priority)
):
    model.train()
    train_meter_c = Meter()
    train_meter_r = Meter()

    # Initialize prior weights for regression tasks (default is all ones; supports user customization)
    if reg_prior_weights is not None:
        reg_prior_weights = torch.tensor(reg_prior_weights, device=args['device'], dtype=torch.float32)
    else:
        if args['task_class'] == 'classification_regression':
            reg_task_num = args['n_tasks'] - args['classification_num']
            reg_prior_weights = torch.ones(reg_task_num, device=args['device'], dtype=torch.float32)
        elif args['task_class'] == 'regression':
            reg_prior_weights = torch.ones(args['n_tasks'], device=args['device'], dtype=torch.float32)

    for batch_id, batch_data in enumerate(data_loader):
        smiles, bg, labels, mask = batch_data
        mask = mask.float().to(args['device'])
        labels = labels.float().to(args['device'])
        atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
        bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
        bg = bg.to(args['device'])
        logits = model(bg, atom_feats, bond_feats, norm=None)
        labels = labels.type_as(logits).to(args['device'])

        if args['task_class'] == 'classification_regression':
            # --------------------- Classification tasks（fixed weights, direct averaging, no dynamic weighting） ---------------------
            logits_c = logits[:, :args['classification_num']]
            labels_c = labels[:, :args['classification_num']]
            mask_c = mask[:, :args['classification_num']]
            class_loss = (loss_criterion_c(logits_c, labels_c) * (mask_c != 0).float()).mean()  # direct averaging

            # --------------------- Regression tasks（dynamic weight = prior weight x loss normalization） ---------------------
            logits_r = logits[:, args['classification_num']:]
            labels_r = labels[:, args['classification_num']:]
            mask_r = mask[:, args['classification_num']:]
            reg_task_num = logits_r.shape[1]

            # Compute the mean loss of each regression task in the current batch (unweighted)
            reg_loss_per_task = torch.mean(
                loss_criterion_r(logits_r, labels_r) * (mask_r != 0).float(), 
                dim=0, keepdim=True  # Average along the sample dimension and keep dimensions (1, reg_task_num)
            ).squeeze()  # Convert to (reg_task_num,)

            # Dynamic weight calculation: prior weight x (current task loss / mean regression-task loss)
            reg_loss_mean = reg_loss_per_task.mean() + 1e-8  # add a small constant to avoid division by zero
            dynamic_reg_weights = reg_prior_weights * (reg_loss_per_task / reg_loss_mean)

            # Compute regression loss: weight each task loss and then average
            reg_loss = (reg_loss_per_task * dynamic_reg_weights).mean()

            total_loss = class_loss + reg_loss

        elif args['task_class'] == 'classification':
            # Classification-only tasks (no regression; keep the original logic)
            class_loss = (loss_criterion_c(logits, labels) * (mask != 0).float()).mean()
            total_loss = class_loss

        elif args['task_class'] == 'regression':
            # --------------------- Regression-only tasks（dynamic weight = prior weight x loss normalization） ---------------------
            reg_task_num = args['n_tasks']
            reg_loss_per_task = torch.mean(
                loss_criterion_r(logits, labels) * (mask != 0).float(), 
                dim=0, keepdim=True
            ).squeeze()
            reg_loss_mean = reg_loss_per_task.mean() + 1e-8
            dynamic_reg_weights = reg_prior_weights * (reg_loss_per_task / reg_loss_mean)
            reg_loss = (reg_loss_per_task * dynamic_reg_weights).mean()
            total_loss = reg_loss

        # --------------------- General training steps (backpropagation and optimizer update) ---------------------
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        # --------------------- Metric recording (classification and regression are handled separately) ---------------------
        if args['task_class'] == 'classification_regression':
            train_meter_c.update(logits_c, labels_c, mask_c)
            train_meter_r.update(logits_r, labels_r, mask_r)
        elif args['task_class'] == 'classification':
            train_meter_c.update(logits, labels, mask)
        else:
            train_meter_r.update(logits, labels, mask)

        # --------------------- Memory cleanup ---------------------
        del smiles, bg, mask, labels, atom_feats, bond_feats, logits
        if args['task_class'] == 'classification_regression':
            del logits_c, labels_c, mask_c, logits_r, labels_r, mask_r
        torch.cuda.empty_cache()

    # --------------------- Print training progress ---------------------
    if args['task_class'] == 'classification_regression':
        train_score = np.mean(
            train_meter_c.compute_metric(args['classification_metric_name']) +
            train_meter_r.compute_metric(args['regression_metric_name'])
        )
        print(f'epoch {epoch + 1}/{args["num_epochs"]}, training r2+auc {train_score:.4f}')
    elif args['task_class'] == 'classification':
        train_score = np.mean(train_meter_c.compute_metric(args['classification_metric_name']))
        print(f'epoch {epoch + 1}/{args["num_epochs"]}, training {args["classification_metric_name"]} {train_score:.4f}')
    else:
        train_score = np.mean(train_meter_r.compute_metric(args['regression_metric_name']))
        print(f'epoch {epoch + 1}/{args["num_epochs"]}, training {args["regression_metric_name"]} {train_score:.4f}')

In [ ]:
def run_an_eval_epoch_heterogeneous(args, model, data_loader):
    model.eval()
    eval_meter_c = Meter()
    eval_meter_r = Meter()
    with torch.no_grad():
        for batch_id, batch_data in enumerate(data_loader):
            smiles, bg, labels, mask = batch_data
            labels = labels.float().to(args['device'])
            mask = mask.float().to(args['device'])
            atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
            bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
            bg = bg.to(args['device'])
            logits = model(bg, atom_feats, bond_feats, norm=None)
            labels = labels.type_as(logits).to(args['device'])
            if args['task_class'] == 'classification_regression':
                # split classification and regression
                logits_c = logits[:, :args['classification_num']]
                labels_c = labels[:, :args['classification_num']]
                mask_c = mask[:, :args['classification_num']]
                logits_r = logits[:, args['classification_num']:]
                labels_r = labels[:, args['classification_num']:]
                mask_r = mask[:, args['classification_num']:]
                # Mask non-existing labels
                eval_meter_c.update(logits_c, labels_c, mask_c)
                eval_meter_r.update(logits_r, labels_r, mask_r)
                del smiles, bg,  mask, labels, atom_feats, bond_feats, logits_c, logits_r, labels_c, labels_r, mask_c, mask_r
                torch.cuda.empty_cache()
            elif args['task_class'] == 'classification':
                # Mask non-existing labels
                eval_meter_c.update(logits, labels, mask)
                del smiles, bg,  mask, labels, atom_feats, bond_feats, logits
                torch.cuda.empty_cache()
            else:
                # Mask non-existing labels
                eval_meter_r.update(logits, labels, mask)
                del smiles, bg,  mask, labels, atom_feats, bond_feats, logits
                torch.cuda.empty_cache()
        if args['task_class'] == 'classification_regression':
            return eval_meter_c.compute_metric(args['classification_metric_name']) + \
                   eval_meter_r.compute_metric(args['regression_metric_name'])
        elif args['task_class'] == 'classification':
            return eval_meter_c.compute_metric(args['classification_metric_name'])
        else:
            return eval_meter_r.compute_metric(args['regression_metric_name'])


def run_an_eval_epoch_pih(args, model, data_loader, output_path):
    model.eval()
    eval_meter_c = Meter()
    eval_meter_r = Meter()
    smiles_list = []
    with torch.no_grad():
        for batch_id, batch_data in enumerate(data_loader):
            smiles, bg, labels, mask = batch_data
            smiles_list = smiles_list + smiles
            labels = labels.float().to(args['device'])
            mask = mask.float().to(args['device'])
            atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
            bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
            bg = bg.to(args['device'])
            logits = model(bg, atom_feats, bond_feats, norm=None)
            labels = labels.type_as(logits).to(args['device'])
            if args['task_class'] == 'classification_regression':
                # split classification and regression
                logits_c = logits[:, :args['classification_num']]
                labels_c = labels[:, :args['classification_num']]
                mask_c = mask[:, :args['classification_num']]
                logits_r = logits[:, args['classification_num']:]
                labels_r = labels[:, args['classification_num']:]
                mask_r = mask[:, args['classification_num']:]
                # Mask non-existing labels
                eval_meter_c.update(logits_c, labels_c, mask_c)
                eval_meter_r.update(logits_r, labels_r, mask_r)
                del smiles, bg,  mask, labels, atom_feats, bond_feats, logits_c, logits_r, labels_c, labels_r, mask_c, mask_r
                torch.cuda.empty_cache()
            elif args['task_class'] == 'classification':
                # Mask non-existing labels
                eval_meter_c.update(logits, labels, mask)
                del smiles, bg,  mask, labels, atom_feats, bond_feats, logits
                torch.cuda.empty_cache()
            else:
                # Mask non-existing labels
                eval_meter_r.update(logits, labels, mask)
                del smiles, bg,  mask, labels, atom_feats, bond_feats, logits
                torch.cuda.empty_cache()
        if args['task_class'] == 'classification_regression':
            return eval_meter_c.compute_metric(args['classification_metric_name']) + \
                   eval_meter_r.compute_metric(args['regression_metric_name'])
        elif args['task_class'] == 'classification':
            y_pred, y_true = eval_meter_c.compute_metric('return_pred_true')
            result = pd.DataFrame(columns=['smiles', 'pred', 'true'])
            result['smiles'] = smiles_list
            result['pred'] = np.squeeze(y_pred.numpy()).tolist()
            result['true'] = np.squeeze(y_true.numpy()).tolist()
            result.to_csv(output_path, index=None)
        else:
            # Get predictions and true labels for regression tasks
            y_pred, y_true = eval_meter_r.compute_metric('return_pred_true')
            # Create a DataFrame to store predictions and true labels
            result = pd.DataFrame(columns=['smiles', 'pred', 'true'])
            # Add SMILES information to the DataFrame
            result['smiles'] = smiles_list
            # Add predictions and true labels to the DataFrame
            result['pred'] = np.squeeze(y_pred.numpy()).tolist()
            result['true'] = np.squeeze(y_true.numpy()).tolist()
            # Write the DataFrame to a CSV file
            result.to_csv(output_path, index=None)
            
def run_an_eval_epoch_re(args, model, data_loader, output_path):
    model.eval()
    eval_meter_r = Meter()
    smiles_list = []
    with torch.no_grad():
        for batch_id, batch_data in enumerate(data_loader):
            smiles, bg, labels, mask = batch_data
            smiles_list = smiles_list + smiles
            labels = labels.float().to(args['device'])
            mask = mask.float().to(args['device'])
            atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
            bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
            bg = bg.to(args['device'])
            logits = model(bg, atom_feats, bond_feats, norm=None).to(args['device'])
            labels = labels.type_as(logits).to(args['device'])
            # Mask non-existing labels
            eval_meter_r.update(logits, labels, mask)
            del smiles, bg, mask, labels, atom_feats, bond_feats, logits
            torch.cuda.empty_cache()

        y_pred, y_true = eval_meter_r.return_pred_true_re()

        result = pd.DataFrame(columns=['smiles', 'pred', 'true'])
        result['smiles'] = smiles_list
        result['pred'] = np.squeeze(y_pred.numpy()).tolist()
        result['true'] = np.squeeze(y_true.numpy()).tolist()
        result.to_csv(output_path, index=None)
        
        return y_pred
        


def run_an_eval_epoch_heterogeneous_return_weight(args, model, data_loader, vis_list=None, vis_task='can'):
    model.eval()
    with torch.no_grad():
        for batch_id, batch_data in enumerate(data_loader):
            smiles, bg, labels, mask = batch_data
            #####
            labels = labels.float().to(args['device'])
            atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
            bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
            bg = bg.to(args['device'])
            logits, atom_weight_list, node_feats = model(bg, atom_feats, bond_feats, norm=None)
            labels = labels.type_as(logits).to(args['device'])
            logits_c = logits[:, :args['classification_num']]
            logits_c = torch.sigmoid(logits_c)
            # different tasks with different atom weight

            for mol_index in range(len(smiles)):
                atom_smiles = smiles[mol_index]
                if atom_smiles in vis_list:
                    for tasks_index in range(1):
                        if args['all_task_list'][tasks_index] == vis_task:
                            if labels[mol_index, tasks_index]!=123456:
                                bg.ndata['w'] = atom_weight_list[tasks_index]
                                bg.ndata['feats'] = node_feats
                                unbatch_bg = dgl.unbatch(bg)
                                one_atom_weight = unbatch_bg[mol_index].ndata['w']
                                one_atom_feats = unbatch_bg[mol_index].ndata['feats']
                                # visual selected molecules
                                print('Tasks:', tasks_index, args['all_task_list'][tasks_index], "**********************")
                                if tasks_index < 26:
                                    print('Predict values:', logits[mol_index, tasks_index])
                                else:
                                    print('Predict values:', logits_c[mol_index, tasks_index])
                                print('True values:', labels[mol_index, tasks_index])
                                weight_visulize(atom_smiles, one_atom_weight)
                    else:
                        continue


def run_an_eval_epoch_heterogeneous_return_weight_py(args, model, data_loader, vis_list=None, vis_task='can'):
    model.eval()
    with torch.no_grad():
        for batch_id, batch_data in enumerate(data_loader):
            smiles, bg, labels, mask = batch_data
            #####
            labels = labels.float().to(args['device'])
            atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
            bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
            bg = bg.to(args['device'])
            logits, atom_weight_list, node_feats = model(bg, atom_feats, bond_feats, norm=None)
            labels = labels.type_as(logits).to(args['device'])
            logits_c = logits[:, :args['classification_num']]
            logits_c = torch.sigmoid(logits_c)
            # different tasks with different atom weight

            for mol_index in range(len(smiles)):
                atom_smiles = smiles[mol_index]
                if atom_smiles in vis_list:
                    for tasks_index in range(1):
                        if args['all_task_list'][tasks_index] == vis_task:
                            if labels[mol_index, tasks_index]!=123456:
                                bg.ndata['w'] = atom_weight_list[tasks_index]
                                bg.ndata['feats'] = node_feats
                                unbatch_bg = dgl.unbatch(bg)
                                one_atom_weight = unbatch_bg[mol_index].ndata['w']
                                one_atom_feats = unbatch_bg[mol_index].ndata['feats']
                                # visual selected molecules
                                print('Tasks:', tasks_index, args['all_task_list'][tasks_index], "**********************")
                                if tasks_index < 26:
                                    print('Predict values:', logits_c[mol_index, tasks_index])
                                else:
                                    print('Predict values:', logits[mol_index, tasks_index])
                                print('True values:', labels[mol_index, tasks_index])
                                weight_visulize_py(atom_smiles, one_atom_weight)
                else:
                    continue


def run_an_eval_epoch_heterogeneous_generate_weight(args, model, data_loader):
    model.eval()
    atom_list_all = []
    with torch.no_grad():
        for batch_id, batch_data in enumerate(data_loader):
            print("batch: {}/{}".format(batch_id+1, len(data_loader)))
            smiles, bg, labels, mask = batch_data
            labels = labels.float().to(args['device'])
            atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
            bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
            bg = bg.to(args['device'])
            logits, atom_weight_list = model(bg, atom_feats, bond_feats, norm=None)
            for atom_weight in atom_weight_list:
                atom_list_all.append(atom_weight[args['select_task_index']])
    task_name = args['select_task_list'][0]
    atom_weight_list = pd.DataFrame(atom_list_all, columns=['atom_weight'])
    atom_weight_list.to_csv(task_name+"_atom_weight.csv", index=None)


def generate_chemical_environment(args, model, data_loader):
    model.eval()
    atom_list_all = []
    with torch.no_grad():
        for batch_id, batch_data in enumerate(data_loader):
            print("batch: {}/{}".format(batch_id + 1, len(data_loader)))
            smiles, bg, labels, mask = batch_data
            print(bg.ndata[args['atom_data_field']][1])
            atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
            bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
            bg = bg.to(args['device'])
            logits, atom_weight_list = model(bg, atom_feats, bond_feats, norm=None)
            print('after training:', bg.ndata['h'][1])


def generate_mol_feats(args, model, data_loader, dataset_output_path):
    model.eval()
    with torch.no_grad():
        for batch_id, batch_data in enumerate(data_loader):
            smiles, bg, labels, mask = batch_data
            bg = bg.to(args['device'])
            atom_feats = bg.ndata.pop(args['atom_data_field']).float().to(args['device'])
            bond_feats = bg.edata.pop(args['bond_data_field']).long().to(args['device'])
            feats = model(bg, atom_feats, bond_feats, norm=None).numpy().tolist()
            feats_name = ['graph-feature' + str(i+1) for i in range(64)]
            data = pd.DataFrame(feats, columns=feats_name)
            data['smiles'] = smiles
            data['labels'] = labels.squeeze().numpy().tolist()
    data.to_csv(dataset_output_path, index=None)


class EarlyStopping(object):
    def __init__(self, pretrained_model='Null_early_stop.pth', mode='higher', patience=10, filename=None, task_name="None"):
        if filename is None:
            task_name = task_name
            filename ='../model/{}_early_stop.pth'.format(task_name)

        assert mode in ['higher', 'lower']
        self.mode = mode
        if self.mode == 'higher':
            self._check = self._check_higher
        else:
            self._check = self._check_lower

        self.patience = patience
        self.counter = 0
        self.filename = filename
        self.best_score = None
        self.early_stop = False
        self.pretrained_model = pretrained_model

    def _check_higher(self, score, prev_best_score):
        return (score > prev_best_score)

    def _check_lower(self, score, prev_best_score):
        return (score < prev_best_score)

    def step(self, score, model):
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(model)
        elif self._check(score, self.best_score):
            self.best_score = score
            self.save_checkpoint(model)
            self.counter = 0
        else:
            self.counter += 1
            print(
                'EarlyStopping counter: {} out of {}'.format(self.counter, self.patience))
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

    def nosave_step(self, score):
        if self.best_score is None:
            self.best_score = score
        elif self._check(score, self.best_score):
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            print(
                'EarlyStopping counter: {} out of {}'.format(self.counter, self.patience))
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

    def save_checkpoint(self, model):
        '''Saves model when the metric on the validation set gets improved.'''
        torch.save({'model_state_dict': model.state_dict()}, self.filename)
        # print(self.filename)

    def load_checkpoint(self, model):
        '''Load model saved with early stopping.'''
        # model.load_state_dict(torch.load(self.filename)['model_state_dict'])
        model.load_state_dict(torch.load(self.filename, map_location=torch.device('cpu'))['model_state_dict'])

    def load_pretrained_model(self, model):
        pretrained_parameters = ['gnn_layers.0.graph_conv_layer.weight',
                                 'gnn_layers.0.graph_conv_layer.h_bias',
                                 'gnn_layers.0.graph_conv_layer.loop_weight',
                                 'gnn_layers.0.res_connection.weight',
                                 'gnn_layers.0.res_connection.bias',
                                 'gnn_layers.0.bn_layer.weight',
                                 'gnn_layers.0.bn_layer.bias',
                                 'gnn_layers.0.bn_layer.running_mean',
                                 'gnn_layers.0.bn_layer.running_var',
                                 'gnn_layers.0.bn_layer.num_batches_tracked',
                                 'gnn_layers.1.graph_conv_layer.weight',
                                 'gnn_layers.1.graph_conv_layer.h_bias',
                                 'gnn_layers.1.graph_conv_layer.loop_weight',
                                 'gnn_layers.1.res_connection.weight',
                                 'gnn_layers.1.res_connection.bias',
                                 'gnn_layers.1.bn_layer.weight',
                                 'gnn_layers.1.bn_layer.bias',
                                 'gnn_layers.1.bn_layer.running_mean',
                                 'gnn_layers.1.bn_layer.running_var',
                                 'gnn_layers.1.bn_layer.num_batches_tracked']
        if torch.cuda.is_available():
            pretrained_model = torch.load('../model/'+self.pretrained_model)
        else:
            pretrained_model = torch.load('../model/'+self.pretrained_model, map_location=torch.device('cpu'))
        model_dict = model.state_dict()
        pretrained_dict = {k: v for k, v in pretrained_model['model_state_dict'].items() if k in pretrained_parameters}
        model_dict.update(pretrained_dict)
        model.load_state_dict(pretrained_dict, strict=False)

    def load_model_attention(self, model):
        pretrained_parameters = ['gnn_layers.0.graph_conv_layer.weight',
                                 'gnn_layers.0.graph_conv_layer.h_bias',
                                 'gnn_layers.0.graph_conv_layer.loop_weight',
                                 'gnn_layers.0.res_connection.weight',
                                 'gnn_layers.0.res_connection.bias',
                                 'gnn_layers.0.bn_layer.weight',
                                 'gnn_layers.0.bn_layer.bias',
                                 'gnn_layers.0.bn_layer.running_mean',
                                 'gnn_layers.0.bn_layer.running_var',
                                 'gnn_layers.0.bn_layer.num_batches_tracked',
                                 'gnn_layers.1.graph_conv_layer.weight',
                                 'gnn_layers.1.graph_conv_layer.h_bias',
                                 'gnn_layers.1.graph_conv_layer.loop_weight',
                                 'gnn_layers.1.res_connection.weight',
                                 'gnn_layers.1.res_connection.bias',
                                 'gnn_layers.1.bn_layer.weight',
                                 'gnn_layers.1.bn_layer.bias',
                                 'gnn_layers.1.bn_layer.running_mean',
                                 'gnn_layers.1.bn_layer.running_var',
                                 'gnn_layers.1.bn_layer.num_batches_tracked',
                                 'weighted_sum_readout.atom_weighting_specific.0.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.0.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.1.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.1.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.2.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.2.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.3.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.3.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.4.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.4.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.5.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.5.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.6.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.6.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.7.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.7.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.8.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.8.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.9.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.9.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.10.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.10.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.11.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.11.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.12.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.12.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.13.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.13.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.14.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.14.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.15.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.15.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.16.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.16.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.17.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.17.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.18.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.18.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.19.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.19.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.20.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.20.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.21.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.21.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.22.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.22.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.23.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.23.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.24.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.24.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.25.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.25.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.26.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.26.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.27.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.27.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.28.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.28.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.29.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.29.0.bias',
                                 'weighted_sum_readout.atom_weighting_specific.30.0.weight',
                                 'weighted_sum_readout.atom_weighting_specific.30.0.bias',
                                 'weighted_sum_readout.shared_weighting.0.weight',
                                 'weighted_sum_readout.shared_weighting.0.bias',
                                 ]
        if torch.cuda.is_available():
            pretrained_model = torch.load('../model/' + self.pretrained_model)
        else:
            pretrained_model = torch.load('../model/' + self.pretrained_model, map_location=torch.device('cpu'))
        model_dict = model.state_dict()
        pretrained_dict = {k: v for k, v in pretrained_model['model_state_dict'].items() if k in pretrained_parameters}
        model_dict.update(pretrained_dict)
        model.load_state_dict(pretrained_dict, strict=False)

In [ ]:
###run code

def load_pretrained_model(self, model):
        pretrained_parameters = ['gnn_layers.0.graph_conv_layer.weight',
                                 'gnn_layers.0.graph_conv_layer.h_bias',
                                 'gnn_layers.0.graph_conv_layer.loop_weight',
                                 'gnn_layers.0.res_connection.weight',
                                 'gnn_layers.0.res_connection.bias',
                                 'gnn_layers.0.bn_layer.weight',
                                 'gnn_layers.0.bn_layer.bias',
                                 'gnn_layers.0.bn_layer.running_mean',
                                 'gnn_layers.0.bn_layer.running_var',
                                 'gnn_layers.0.bn_layer.num_batches_tracked',
                                 'gnn_layers.1.graph_conv_layer.weight',
                                 'gnn_layers.1.graph_conv_layer.h_bias',
                                 'gnn_layers.1.graph_conv_layer.loop_weight',
                                 'gnn_layers.1.res_connection.weight',
                                 'gnn_layers.1.res_connection.bias',
                                 'gnn_layers.1.bn_layer.weight',
                                 'gnn_layers.1.bn_layer.bias',
                                 'gnn_layers.1.bn_layer.running_mean',
                                 'gnn_layers.1.bn_layer.running_var',
                                 'gnn_layers.1.bn_layer.num_batches_tracked']
        
        if torch.cuda.is_available():
            pretrained_model = torch.load('../model/'+self.pretrained_model)
        else:
            pretrained_model = torch.load('../model/'+self.pretrained_model, map_location=torch.device('cpu'))
        
        model_dict = model.state_dict()
        # Filter out parameters with mismatched shapes
        pretrained_dict = {k: v for k, v in pretrained_model['model_state_dict'].items() 
                           if k in pretrained_parameters and v.shape == model_dict[k].shape}
        
        print(f"Loaded {len(pretrained_dict)}/{len(pretrained_parameters)} pretrained parameters")
        if len(pretrained_dict) < len(pretrained_parameters):
            print("The following parameters were not loaded because of shape mismatch:")
            for k in pretrained_parameters:
                if k not in pretrained_dict:
                    print(f"  - {k} (pretrained shape: {pretrained_model['model_state_dict'][k].shape}, current model shape: {model_dict[k].shape})")
        
        model_dict.update(pretrained_dict)
        model.load_state_dict(pretrained_dict, strict=False)

# Molecular graph-based mechanistic interpretation

In [ ]:
# Diagnostic code: inspect model file contents
import torch
model_save_path = r'D:\LTY\DW-MTL - copy\DW-MTL-noncancerbest_model\noncancerCYP450class_run10_20260212_234758.pth'

# Load file
checkpoint = torch.load(model_save_path, map_location='cpu')

print(f"File type: {type(checkpoint)}")
print(f"File attributes: {dir(checkpoint) if hasattr(checkpoint, '__dict__') else 'not an object'}")

# If it is a dictionary, print the keys
if isinstance(checkpoint, dict):
    print("Dictionary keys:", checkpoint.keys())
    for key in checkpoint.keys():
        print(f"  {key}: {type(checkpoint[key])}")
        
# If it is a model object, inspect its attributes
if hasattr(checkpoint, 'state_dict'):
    print("This is a model object, and state_dict() can be called")

In [ ]:
# Model prediction + molecular graph mechanistic interpretation
import torch
import pickle as pkl
import pandas as pd
import numpy as np
import os
import dgl
from torch.utils.data import DataLoader
from tqdm import tqdm

# Import the MGA class and graph-construction functions
import sys
sys.path.append(r'D:\code\transfer_learning\DW-MTL\utils')

try:
    from MY_GNN import MGA
    print("Successfully imported MGA")
    
    def collate_molgraphs(data):
        smiles, graphs, labels, mask = map(list, zip(*data))
        bg = dgl.batch(graphs)
        bg.set_n_initializer(dgl.init.zero_initializer)
        bg.set_e_initializer(dgl.init.zero_initializer)
        labels = torch.tensor(labels)
        mask = torch.tensor(mask)
        return smiles, bg, labels, mask
        
    print("Redefined the collate_molgraphs function (on CPU)")
    
except ImportError as e:
    print(f"Failed to import MGA: {e}")
    def collate_molgraphs(data):
        smiles, graphs, labels, mask = map(list, zip(*data))
        bg = dgl.batch(graphs)
        bg.set_n_initializer(dgl.init.zero_initializer)
        bg.set_e_initializer(dgl.init.zero_initializer)
        labels = torch.tensor(labels)
        mask = torch.tensor(mask)
        return smiles, bg, labels, mask

# Set environment variables to prevent CUDA usage
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

# Define model and hyperparameter paths
model_save_path = r'D:\LTY\DW-MTL - copy\DW-MTL-cancerbest_model\DW-MTL-cancer.pth'
hyperparameters_save_path = r'D:\LTY\DW-MTL - copy\DW-MTL-cancerbest_model\DW-MTL-cancer.pkl'

print("Checking CUDA environment...")
print(f"CUDA available: {torch.cuda.is_available()}")

print("Loading hyperparameters...")
with open(hyperparameters_save_path, 'rb') as f:
    args = pkl.load(f)

# Determine task information
if 'select_task_list' in args:
    task_names = args['select_task_list']
    n_tasks = len(task_names)
else:
    task_names = ['APO', 'ELM', 'EPM', 'GIN', 'PRO', 'OXP', 'ISF', 'IUR', 'OSF']
    n_tasks = 9
    args['classification_num'] = 6
    args['regression_num'] = 3

if 'in_feats' in args:
    expected_feature_dim = args['in_feats']
else:
    expected_feature_dim = 40
    args['in_feats'] = expected_feature_dim

device = 'cpu'
args['device'] = device

# Modified section: reinitialize the model and load weights
print("\nLoading model...")
try:
    # 1. Reinitialize the model and set return_weight=True
    model = MGA(in_feats=args['in_feats'], 
                rgcn_hidden_feats=args['rgcn_hidden_feats'],
                n_tasks=n_tasks,
                return_weight=True,  # Key setting: set to True to return atom weights
                classifier_hidden_feats=args['classifier_hidden_feats'], 
                loop=args['loop'],
                return_mol_embedding=False,  # Key setting: set to False
                rgcn_drop_out=args['rgcn_drop_out'],
                dropout=args['drop_out'])
    
    # 2. Loading model weights
    checkpoint = torch.load(model_save_path, map_location=torch.device('cpu'))
    
    # Check checkpoint type
    print(f"Checkpoint type: {type(checkpoint)}")
    
    if isinstance(checkpoint, MGA):
        # If it is a complete MGA object, use its state dictionary directly
        print("Loaded a complete MGA model object")
        model.load_state_dict(checkpoint.state_dict())
    elif isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        # If it is a dictionary containing model_state_dict
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        # If it is a state dictionary in another format
        model.load_state_dict(checkpoint)
    
    # 3. Ensure model attributes are set correctly
    model.return_weight = True
    model.return_mol_embedding = False
    
    model.to(device)
    model.eval()
    
    print("Model loaded successfully!")
    print(f"Model return_weight attribute: {model.return_weight}")
    print(f"Model return_mol_embedding attribute: {model.return_mol_embedding}")
    
except Exception as e:
    print(f"Model loading failed: {e}")
    import traceback
    traceback.print_exc()
    exit()

print("\n" + "=" * 50)
print("Model loading completed!")
print("=" * 50)

# Import RDKit for molecular visualization
try:
    from rdkit import Chem
    from rdkit.Chem.Draw import rdMolDraw2D
    from rdkit.Chem import rdDepictor
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib import cm
    RDKIT_AVAILABLE = True
    print("\nRDKit imported successfully!")
except ImportError as e:
    print(f"Failed to import RDKit: {e}")
    print("Atom-weight visualization will be unavailable")
    RDKIT_AVAILABLE = False

def one_of_k_encoding(x, allowable_set):
    if x not in allowable_set:
        raise Exception("input {0} not in allowable set{1}:".format(x, allowable_set))
    return [x == s for s in allowable_set]

def one_of_k_encoding_unk(x, allowable_set):
    if x not in allowable_set:
        x = allowable_set[-1]
    return [x == s for s in allowable_set]

def atom_features(atom, explicit_H=False, use_chirality=True):
    results = one_of_k_encoding_unk(
        atom.GetSymbol(),
        ['B', 'C', 'N', 'O', 'F', 'Si', 'P', 'S', 'Cl', 'As', 
         'Se', 'Br', 'Te', 'I', 'At', 'other']
    ) + one_of_k_encoding(atom.GetDegree(), [0, 1, 2, 3, 4, 5, 6]) + \
    [atom.GetFormalCharge(), atom.GetNumRadicalElectrons()] + \
    one_of_k_encoding_unk(atom.GetHybridization(), [
        Chem.rdchem.HybridizationType.SP, 
        Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3, 
        Chem.rdchem.HybridizationType.SP3D,
        Chem.rdchem.HybridizationType.SP3D2, 'other']
    ) + [atom.GetIsAromatic()]
    
    if not explicit_H:
        results = results + one_of_k_encoding_unk(atom.GetTotalNumHs(), [0, 1, 2, 3, 4])
    
    if use_chirality:
        try:
            results = results + one_of_k_encoding_unk(
                atom.GetProp('_CIPCode'), ['R', 'S']
            ) + [atom.HasProp('_ChiralityPossible')]
        except:
            results = results + [False, False] + [atom.HasProp('_ChiralityPossible')]

    return np.array(results, dtype=np.float32)

def one_of_k_atompair_encoding(x, allowable_set):
    for atompair in allowable_set:
        if x in atompair:
            x = atompair
            break
        else:
            if atompair == allowable_set[-1]:
                x = allowable_set[-1]
            else:
                continue
    return [x == s for s in allowable_set]

def etype_features(bond, use_chirality=True, atompair=True):
    bt = bond.GetBondType()
    bond_feats_1 = [
        bt == Chem.rdchem.BondType.SINGLE, 
        bt == Chem.rdchem.BondType.DOUBLE,
        bt == Chem.rdchem.BondType.TRIPLE, 
        bt == Chem.rdchem.BondType.AROMATIC,
    ]
    
    for i, m in enumerate(bond_feats_1):
        if m == True:
            a = i

    bond_feats_2 = bond.GetIsConjugated()
    if bond_feats_2 == True:
        b = 1
    else:
        b = 0

    bond_feats_3 = bond.IsInRing()
    if bond_feats_3 == True:
        c = 1
    else:
        c = 0

    index = a * 1 + b * 4 + c * 8
    
    if use_chirality:
        bond_feats_4 = one_of_k_encoding_unk(
            str(bond.GetStereo()),
            ["STEREONONE", "STEREOANY", "STEREOZ", "STEREOE"]
        )
        for i, m in enumerate(bond_feats_4):
            if m == True:
                d = i
        index = index + d * 16
    
    if atompair == True:
        atom_pair_str = bond.GetBeginAtom().GetSymbol() + bond.GetEndAtom().GetSymbol()
        bond_feats_5 = one_of_k_atompair_encoding(
            atom_pair_str, [
                ['CC'], ['CN', 'NC'], ['ON', 'NO'], ['CO', 'OC'], ['CS', 'SC'],
                ['SO', 'OS'], ['NN'], ['SN', 'NS'], ['CCl', 'ClC'], ['CF', 'FC'],
                ['CBr', 'BrC'], ['others']
            ]
        )
        for i, m in enumerate(bond_feats_5):
            if m == True:
                e = i
        index = index + e * 64
    
    return index

def construct_RGCN_bigraph_from_smiles(smiles):
    try:
        g = dgl.graph([])
        
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError(f"Unable to create molecule from SMILES: {smiles}")
        
        mol = Chem.AddHs(mol)
        
        num_atoms = mol.GetNumAtoms()
        g.add_nodes(num_atoms)
        
        atoms_feature_all = []
        for atom_index, atom in enumerate(mol.GetAtoms()):
            atom_feature = atom_features(atom, explicit_H=False, use_chirality=True)
            atoms_feature_all.append(torch.from_numpy(atom_feature))
        
        feature_dim = atoms_feature_all[0].shape[0]
        
        if feature_dim != expected_feature_dim:
            if feature_dim < expected_feature_dim:
                for i in range(len(atoms_feature_all)):
                    padding = torch.zeros(expected_feature_dim - feature_dim)
                    atoms_feature_all[i] = torch.cat([atoms_feature_all[i], padding])
            else:
                for i in range(len(atoms_feature_all)):
                    atoms_feature_all[i] = atoms_feature_all[i][:expected_feature_dim]
        
        g.ndata["atom"] = torch.stack(atoms_feature_all)
        
        src_list = []
        dst_list = []
        etype_feature_all = []
        
        num_bonds = mol.GetNumBonds()
        for i in range(num_bonds):
            bond = mol.GetBondWithIdx(i)
            etype_feature = etype_features(bond, use_chirality=True, atompair=True)
            u = bond.GetBeginAtomIdx()
            v = bond.GetEndAtomIdx()
            src_list.extend([u, v])
            dst_list.extend([v, u])
            etype_feature_all.append(etype_feature)
            etype_feature_all.append(etype_feature)
        
        if src_list:
            g.add_edges(src_list, dst_list)
            
            normal_all = []
            for i in etype_feature_all:
                normal = etype_feature_all.count(i)/len(etype_feature_all)
                normal = round(normal, 1)
                normal_all.append(normal)
            
            g.edata["etype"] = torch.tensor(etype_feature_all, dtype=torch.long)
            g.edata["normal"] = torch.tensor(normal_all, dtype=torch.float)
        else:
            g.edata["etype"] = torch.tensor([], dtype=torch.long)
            g.edata["normal"] = torch.tensor([], dtype=torch.float)
        
        return g
        
    except Exception as e:
        print(f"Failed to construct molecular graph: {e}")
        raise

def weight_visulize_py2_custom(smiles, atom_weight, save_path, task_name="Task", size=(500, 500)):
    """Visualize atom weights using continuous color mapping only."""
    if not RDKIT_AVAILABLE:
        print("Warning: RDKit is unavailable; visualization cannot be generated")
        return None

    try:
        if hasattr(atom_weight, 'cpu'):
            atom_weight_list = atom_weight.squeeze().cpu().numpy()
        else:
            atom_weight_list = atom_weight.squeeze().numpy()

        if np.ndim(atom_weight_list) == 0:
            atom_new_weight = np.array([float(atom_weight_list)], dtype=float)
        else:
            atom_new_weight = np.asarray(atom_weight_list, dtype=float).flatten()

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"Warning: unable to create a molecule from SMILES '{smiles}'")
            return None

        mol = Chem.AddHs(mol)
        rdDepictor.Compute2DCoords(mol)

        if len(atom_new_weight) > 0:
            weight_min = float(np.min(atom_new_weight))
            weight_max = float(np.max(atom_new_weight))
            if weight_max > weight_min:
                atom_new_weight = (atom_new_weight - weight_min) / (weight_max - weight_min)
            else:
                atom_new_weight = np.full_like(atom_new_weight, 0.5, dtype=float)

        if mol.GetNumAtoms() > 0 and len(atom_new_weight) != mol.GetNumAtoms():
            print(
                f"Warning: the number of atom weights ({len(atom_new_weight)}) does not match "
                f"the number of atoms ({mol.GetNumAtoms()}); using a neutral weight of 0.5."
            )
            atom_new_weight = np.full(mol.GetNumAtoms(), 0.5, dtype=float)

        norm = matplotlib.colors.Normalize(vmin=0, vmax=1)
        cmap = cm.get_cmap('Oranges')
        plt_colors = cm.ScalarMappable(norm=norm, cmap=cmap)

        atom_colors = {i: plt_colors.to_rgba(float(atom_new_weight[i])) for i in range(mol.GetNumAtoms())}
        bond_colors = {}
        for i in range(mol.GetNumBonds()):
            bond = mol.GetBondWithIdx(i)
            u = bond.GetBeginAtomIdx()
            v = bond.GetEndAtomIdx()
            if u < len(atom_new_weight) and v < len(atom_new_weight):
                bond_weight = (atom_new_weight[u] + atom_new_weight[v]) / 2
                bond_colors[i] = plt_colors.to_rgba(float(abs(bond_weight)))
            else:
                bond_colors[i] = plt_colors.to_rgba(0.0)

        drawer = rdMolDraw2D.MolDraw2DSVG(size[0], size[1])
        drawer.SetFontSize(1)
        mol = rdMolDraw2D.PrepareMolForDrawing(mol)
        drawer.DrawMolecule(
            mol,
            highlightAtoms=list(range(mol.GetNumAtoms())),
            highlightBonds=list(range(mol.GetNumBonds())),
            highlightAtomColors=atom_colors,
            highlightBondColors=bond_colors
        )
        drawer.FinishDrawing()

        svg = drawer.GetDrawingText()
        svg_path = f"{save_path}.svg"
        with open(svg_path, 'w', encoding='utf-8') as f:
            f.write(svg)

        print(f"Atom-weight visualization saved to: {svg_path}")
        return svg_path

    except Exception as e:
        print(f"Visualization generation failed: {e}")
        import traceback
        traceback.print_exc()
        return None


def create_molecule_dataset(smiles_list):
    print(f"\nProcessing {len(smiles_list)} molecules...")
    dataset = []
    failed_smiles = []
    
    for i, smiles in enumerate(tqdm(smiles_list, desc="Constructing molecular graphs")):
        try:
            g = construct_RGCN_bigraph_from_smiles(smiles)
            labels = [0.0] * n_tasks
            mask = [1.0] * n_tasks
            dataset.append((smiles, g, labels, mask))
        except Exception as e:
            print(f"Molecule {i+1}: {smiles} conversion failed - {e}")
            failed_smiles.append(smiles)
    
    print(f"Successfully processed {len(dataset)} molecules; failed {len(failed_smiles)} ")
    if failed_smiles:
        print("Failed SMILES (first 10):", failed_smiles[:10])
    
    return dataset, failed_smiles

def predict_with_atom_weights(model, data_loader, output_dir='./prediction_results', 
                             task_names=None, batch_size=32, visualize_all_tasks=True, 
                             specific_task_idx=None, visualize_method="continuous"):
    """
    The visualization method is fixed to continuous atom-weight mapping.
    """
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'visualizations'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'atom_weights'), exist_ok=True)
    
    model.eval()
    all_predictions = []
    all_smiles = []
    all_atom_weights = []
    
    print(f"Starting prediction; results will be saved to: {output_dir}")
    print(f"Visualization method: {visualize_method}")
    
    batch_counter = 0
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(data_loader, desc="Prediction progress")):
            smiles, bg, labels, mask = batch
            
            bg = bg.to('cpu')
            
            try:
                atom_feats = bg.ndata.pop('atom').float().to('cpu')
            except KeyError:
                if 'h' in bg.ndata:
                    atom_feats = bg.ndata.pop('h').float().to('cpu')
                elif 'feat' in bg.ndata:
                    atom_feats = bg.ndata.pop('feat').float().to('cpu')
                else:
                    raise ValueError("Atom features not found")
            
            try:
                bond_feats = bg.edata.pop('etype').long().to('cpu')
            except KeyError:
                if 'bond_type' in bg.edata:
                    bond_feats = bg.edata.pop('bond_type').long().to('cpu')
                else:
                    bond_feats = torch.zeros(bg.num_edges(), dtype=torch.long).to('cpu')
            
            try:
                print(f"\nBatch {batch_idx+1}: running model prediction...")
                print(f"  Input: atom feature shape: {atom_feats.shape}, bond feature shape: {bond_feats.shape}")
                
                # Call the model; according to MY_GNN.py, three values are returned when return_weight=True
                predictions, atom_weight_list, node_feats = model(bg, atom_feats, bond_feats, norm=None)
                
                print(f"  Prediction shape: {predictions.shape}")
                print(f"  Length of atom-weight list: {len(atom_weight_list)}")
                print(f"  Node feature shape: {node_feats.shape}")
                
                # Check atom weights
                if isinstance(atom_weight_list, list) and len(atom_weight_list) > 0:
                    print(f"  Atom-weight shape for the first task: {atom_weight_list[0].shape}")
                    print(f"  Atom-weight value range for the first task: [{atom_weight_list[0].min():.4f}, {atom_weight_list[0].max():.4f}]")
                    
            except Exception as e:
                print(f"Batch {batch_idx+1}: model prediction failed - {e}")
                import traceback
                traceback.print_exc()
                raise
            
            classification_num = args.get('classification_num', 6)
            
            if predictions.shape[1] == n_tasks:
                predictions_c = torch.sigmoid(predictions[:, :classification_num])
                predictions_r = predictions[:, classification_num:]
                batch_pred = torch.cat([predictions_c, predictions_r], dim=1)
            else:
                batch_pred = predictions
            
            unbatch_bg = dgl.unbatch(bg)
            
            for mol_idx in range(len(smiles)):
                mol_smiles = smiles[mol_idx]
                mol_graph = unbatch_bg[mol_idx]
                mol_atom_count = mol_graph.number_of_nodes()
                
                start_idx = 0
                for i in range(mol_idx):
                    start_idx += unbatch_bg[i].number_of_nodes()
                end_idx = start_idx + mol_atom_count
                
                mol_atom_weights_by_task = []
                for task_idx in range(len(task_names)):
                    if task_idx < len(atom_weight_list):
                        task_weights = atom_weight_list[task_idx][start_idx:end_idx]
                        mol_atom_weights_by_task.append(task_weights)
                    else:
                        # If the number of tasks exceeds the atom-weight list length, use the average weight
                        mol_atom_weights_by_task.append(torch.ones(mol_atom_count) * 0.5)
                
                all_smiles.append(mol_smiles)
                all_predictions.append(batch_pred[mol_idx].cpu().numpy())
                all_atom_weights.append(mol_atom_weights_by_task)
                
                if RDKIT_AVAILABLE and (visualize_all_tasks or specific_task_idx is not None):
                    if specific_task_idx is not None:
                        tasks_to_visualize = [specific_task_idx]
                    else:
                        tasks_to_visualize = list(range(len(task_names)))
                    
                    for task_idx in tasks_to_visualize:
                        if task_idx < len(mol_atom_weights_by_task):
                            task_weights = mol_atom_weights_by_task[task_idx]
                            task_name = task_names[task_idx]
                            
                            if hasattr(task_weights, 'cpu'):
                                weights_np = task_weights.cpu().numpy()
                            else:
                                weights_np = task_weights.numpy()
                            
                            if len(weights_np) > 0:
                                # Save atom-weight data
                                weight_df = pd.DataFrame({
                                    'atom_index': range(len(weights_np)),
                                    'weight': weights_np.flatten()
                                })
                                
                                safe_smiles = mol_smiles.replace('/', '_').replace('\\', '_').replace(':', '_')
                                if len(safe_smiles) > 50:
                                    safe_smiles = safe_smiles[:50]
                                
                                weight_path = os.path.join(output_dir, 'atom_weights', 
                                                          f'mol{batch_counter:04d}_{task_name}_weights.csv')
                                weight_df.to_csv(weight_path, index=False)
                                
                                # Generate visualization
                                save_path = os.path.join(output_dir, 'visualizations', 
                                                        f'mol{batch_counter:04d}_{task_name}_{visualize_method}')
                                svg_path = weight_visulize_py2_custom(
                                    mol_smiles, task_weights, save_path, 
                                    task_name=task_name, )
                
                batch_counter += 1
    
    if len(all_predictions) > 0:
        results_df = pd.DataFrame()
        results_df['SMILES'] = all_smiles
        
        for task_idx, task_name in enumerate(task_names):
            if task_idx < len(all_predictions[0]):
                results_df[task_name] = [pred[task_idx] for pred in all_predictions]
        
        csv_path = os.path.join(output_dir, 'predictions.csv')
        results_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f"\nAll prediction results saved to: {csv_path}")
        
        summary_path = os.path.join(output_dir, 'prediction_summary.txt')
        with open(summary_path, 'w') as f:
            f.write(f"Prediction summary\n")
            f.write(f"==============\n")
            f.write(f"Total number of molecules: {len(all_smiles)}\n")
            f.write(f"Number of tasks: {len(task_names)}\n")
            f.write(f"Classification tasks: {task_names[:args.get('classification_num', 6)]}\n")
            f.write(f"Regression tasks: {task_names[args.get('classification_num', 6):]}\n")
            f.write(f"Input feature dimension: {args.get('in_feats', 'unknown')}\n")
            f.write(f"Output directory: {output_dir}\n")
            f.write(f"Visualization method: {visualize_method}\n")
        
        print(f"Summary information saved to: {summary_path}")
        
        print(f"\nPrediction results for the first five molecules:")
        for i in range(min(5, len(all_smiles))):
            print(f"\nMolecule {i+1}: {all_smiles[i]}")
            for task_idx, task_name in enumerate(task_names):
                if task_idx < len(all_predictions[i]):
                    print(f"  {task_name}: {all_predictions[i][task_idx]:.6f}")
    
    return {
        'smiles': all_smiles,
        'predictions': all_predictions,
        'atom_weights': all_atom_weights,
        'results_df': results_df if 'results_df' in locals() else None
    }

# Main program
def main():
    print("=" * 60)
    print("Multi-task model prediction and atom-weight visualization tool")
    print("=" * 60)
    
    visualize_method = "continuous"
    print(f"Using visualization method: {visualize_method}")
    
    # 1. Read data
    print("\n1. Reading data...")
    data_path = r"D:\LTY\DW-MTL - copy\data\TOP20_compounds\ISF.csv"
    try:
        new = pd.read_csv(data_path)
        new_data_smiles = new.iloc[:, 0].values.tolist()
        print(f"Successfully read all SMILES from {data_path} read {len(new_data_smiles)} SMILES")
        
        if len(new_data_smiles) > 5:
            new_data_smiles = new_data_smiles[:5]
            print("Test only the first five molecules")
    except Exception as e:
        print(f"Failed to read data: {e}")
        new_data_smiles = [
            'CC(=O)OC1=CC=CC=C1C(=O)O',  # aspirin
            'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',  # caffeine
            'CC(C)CC1=CC=C(C=C1)C(C)C(=O)O',  # ibuprofen
        ]
    
    # 2. Constructing molecular graph dataset
    print("\n2. Constructing molecular graph dataset...")
    dataset, failed_smiles = create_molecule_dataset(new_data_smiles)
    
    if len(dataset) == 0:
        print("Error: no molecular graphs were successfully constructed")
        return
    
    # 3. Create data loader
    print("\n3. Create data loader...")
    batch_size = min(32, len(dataset))
    data_loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate_molgraphs, shuffle=False)
    
    # 4. Set visualization options
    print("\n4. Set visualization options...")
    visualize_all_tasks = False
    specific_task_idx = 0
    print(f"Visualize all tasks: {visualize_all_tasks}")
    print(f"Specific task index: {specific_task_idx} ({task_names[specific_task_idx]})")
    
    # 5. Run prediction and visualization
    print("\n5. Starting prediction and visualization...")
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"./prediction_results_{timestamp}"
    
    results = predict_with_atom_weights(
        model=model,
        data_loader=data_loader,
        output_dir=output_dir,
        task_names=task_names,
        batch_size=batch_size,
        visualize_all_tasks=visualize_all_tasks,
        specific_task_idx=specific_task_idx,
        visualize_method=visualize_method
    )
    
    print("\n" + "=" * 60)
    print("Prediction completed!")
    print(f"Results saved in: {output_dir}")
    print(f"Number of successfully predicted molecules: {len(results['smiles'])}")
    if failed_smiles:
        print(f"Number of failed molecules: {len(failed_smiles)}")
    print("=" * 60)

if __name__ == "__main__":
    main()


In [ ]:
# ==============================
# Multi-task model prediction + atom-weight molecular graph mechanistic interpretation
# Notes:
# 1. Visualize the ISF / IUR / OSF tasks simultaneously
# 2. Analyze all SMILES in the CSV file instead of only the first five
# 3. Automatically identify common SMILES column names; if no match is found, use the first column by default
# ==============================

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'   # Force CPU usage

import sys
import torch
import pickle as pkl
import pandas as pd
import numpy as np
import dgl

from torch.utils.data import DataLoader
from tqdm import tqdm
from datetime import datetime

# ==============================
# Path settings
# ==============================
sys.path.append(r'D:\code\transfer_learning\DW-MTL\utils')

model_save_path = r'D:\LTY\DW-MTL - copy\DW-MTL-noncancerbest_model\noncancerCYP450class_run10_20260212_234758.pth'
hyperparameters_save_path = r'D:\LTY\DW-MTL - copy\DW-MTL-noncancerbest_model\noncancerCYP450class_hyperparameters_run10_20260212_234758.pkl'

# Change this to your own CSV path
data_path = r"D:\LTY\DW-MTL - copy\data\electroplating_park_PFAS_list.csv"

# If you know the SMILES column name, specify it directly, such as 'SMILES' or 'canonical_smiles'
# If set to None, the program will identify it automatically; if no match is found, the first column will be used by default
smiles_col = None

# Target visualization tasks
# target_task_names = ['rfd', 'rfc', 'bmd']
target_task_names = ['bmd']
# ==============================
# Import model
# ==============================
try:
    from MY_GNN import MGA
    print("Successfully imported MGA")

    def collate_molgraphs(data):
        smiles, graphs, labels, mask = map(list, zip(*data))
        bg = dgl.batch(graphs)
        bg.set_n_initializer(dgl.init.zero_initializer)
        bg.set_e_initializer(dgl.init.zero_initializer)
        labels = torch.tensor(labels)
        mask = torch.tensor(mask)
        return smiles, bg, labels, mask

    print("Defined collate_molgraphs (CPU version)")

except ImportError as e:
    print(f"Failed to import MGA: {e}")

    def collate_molgraphs(data):
        smiles, graphs, labels, mask = map(list, zip(*data))
        bg = dgl.batch(graphs)
        bg.set_n_initializer(dgl.init.zero_initializer)
        bg.set_e_initializer(dgl.init.zero_initializer)
        labels = torch.tensor(labels)
        mask = torch.tensor(mask)
        return smiles, bg, labels, mask


print("Checking CUDA environment...")
print(f"CUDA available: {torch.cuda.is_available()}")

print("Loading hyperparameters...")
with open(hyperparameters_save_path, 'rb') as f:
    args = pkl.load(f)

# ==============================
# Task information
# ==============================
if 'select_task_list' in args:
    task_names = args['select_task_list']
    n_tasks = len(task_names)
else:
    task_names = ['1a2', '2c9', '2c19', '2d6', '3a4', 'rfd', 'rfc', 'bmd']
    n_tasks = 8
    args['classification_num'] = 5
    args['regression_num'] = 3

if 'in_feats' in args:
    expected_feature_dim = args['in_feats']
else:
    expected_feature_dim = 40
    args['in_feats'] = expected_feature_dim

device = 'cpu'
args['device'] = device

# ==============================
# Loading model
# ==============================
print("\nLoading model...")
try:
    model = MGA(
        in_feats=args['in_feats'],
        rgcn_hidden_feats=args['rgcn_hidden_feats'],
        n_tasks=n_tasks,
        return_weight=True,
        classifier_hidden_feats=args['classifier_hidden_feats'],
        loop=args['loop'],
        return_mol_embedding=False,
        rgcn_drop_out=args['rgcn_drop_out'],
        dropout=args['drop_out']
    )

    checkpoint = torch.load(model_save_path, map_location=torch.device('cpu'))
    print(f"Checkpoint type: {type(checkpoint)}")

    if isinstance(checkpoint, MGA):
        print("Loaded a complete MGA model object")
        model.load_state_dict(checkpoint.state_dict())
    elif isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)

    model.return_weight = True
    model.return_mol_embedding = False
    model.to(device)
    model.eval()

    print("Model loaded successfully!")
    print(f"model.return_weight = {model.return_weight}")
    print(f"model.return_mol_embedding = {model.return_mol_embedding}")

except Exception as e:
    print(f"Model loading failed: {e}")
    import traceback
    traceback.print_exc()
    raise

print("\n" + "=" * 60)
print("Model loading completed")
print("=" * 60)

# ==============================
# Import RDKit and Matplotlib
# ==============================
try:
    from rdkit import Chem
    from rdkit.Chem.Draw import rdMolDraw2D
    from rdkit.Chem import rdDepictor
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib import cm
    RDKIT_AVAILABLE = True
    print("RDKit imported successfully!")
except ImportError as e:
    print(f"Failed to import RDKit: {e}")
    print("Atom-weight visualization is unavailable")
    RDKIT_AVAILABLE = False


# ==============================
# Feature engineering functions
# ==============================
def one_of_k_encoding(x, allowable_set):
    if x not in allowable_set:
        raise Exception(f"input {x} not in allowable set {allowable_set}")
    return [x == s for s in allowable_set]


def one_of_k_encoding_unk(x, allowable_set):
    if x not in allowable_set:
        x = allowable_set[-1]
    return [x == s for s in allowable_set]


def atom_features(atom, explicit_H=False, use_chirality=True):
    results = one_of_k_encoding_unk(
        atom.GetSymbol(),
        ['B', 'C', 'N', 'O', 'F', 'Si', 'P', 'S', 'Cl', 'As',
         'Se', 'Br', 'Te', 'I', 'At', 'other']
    ) + one_of_k_encoding(atom.GetDegree(), [0, 1, 2, 3, 4, 5, 6]) + \
    [atom.GetFormalCharge(), atom.GetNumRadicalElectrons()] + \
    one_of_k_encoding_unk(atom.GetHybridization(), [
        Chem.rdchem.HybridizationType.SP,
        Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3,
        Chem.rdchem.HybridizationType.SP3D,
        Chem.rdchem.HybridizationType.SP3D2,
        'other'
    ]) + [atom.GetIsAromatic()]

    if not explicit_H:
        results = results + one_of_k_encoding_unk(atom.GetTotalNumHs(), [0, 1, 2, 3, 4])

    if use_chirality:
        try:
            results = results + one_of_k_encoding_unk(
                atom.GetProp('_CIPCode'), ['R', 'S']
            ) + [atom.HasProp('_ChiralityPossible')]
        except Exception:
            results = results + [False, False] + [atom.HasProp('_ChiralityPossible')]

    return np.array(results, dtype=np.float32)


def one_of_k_atompair_encoding(x, allowable_set):
    for atompair in allowable_set:
        if x in atompair:
            x = atompair
            break
        else:
            if atompair == allowable_set[-1]:
                x = allowable_set[-1]
    return [x == s for s in allowable_set]


def etype_features(bond, use_chirality=True, atompair=True):
    bt = bond.GetBondType()
    bond_feats_1 = [
        bt == Chem.rdchem.BondType.SINGLE,
        bt == Chem.rdchem.BondType.DOUBLE,
        bt == Chem.rdchem.BondType.TRIPLE,
        bt == Chem.rdchem.BondType.AROMATIC,
    ]

    for i, m in enumerate(bond_feats_1):
        if m:
            a = i
            break

    bond_feats_2 = bond.GetIsConjugated()
    b = 1 if bond_feats_2 else 0

    bond_feats_3 = bond.IsInRing()
    c = 1 if bond_feats_3 else 0

    index = a * 1 + b * 4 + c * 8

    if use_chirality:
        bond_feats_4 = one_of_k_encoding_unk(
            str(bond.GetStereo()),
            ["STEREONONE", "STEREOANY", "STEREOZ", "STEREOE"]
        )
        for i, m in enumerate(bond_feats_4):
            if m:
                d = i
                break
        index = index + d * 16

    if atompair:
        atom_pair_str = bond.GetBeginAtom().GetSymbol() + bond.GetEndAtom().GetSymbol()
        bond_feats_5 = one_of_k_atompair_encoding(
            atom_pair_str,
            [
                ['CC'], ['CN', 'NC'], ['ON', 'NO'], ['CO', 'OC'], ['CS', 'SC'],
                ['SO', 'OS'], ['NN'], ['SN', 'NS'], ['CCl', 'ClC'], ['CF', 'FC'],
                ['CBr', 'BrC'], ['others']
            ]
        )
        for i, m in enumerate(bond_feats_5):
            if m:
                e = i
                break
        index = index + e * 64

    return index


def construct_RGCN_bigraph_from_smiles(smiles):
    try:
        g = dgl.graph([])

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError(f"Unable to create molecule from SMILES: {smiles}")

        mol = Chem.AddHs(mol)
        num_atoms = mol.GetNumAtoms()
        g.add_nodes(num_atoms)

        atoms_feature_all = []
        for atom in mol.GetAtoms():
            atom_feature = atom_features(atom, explicit_H=False, use_chirality=True)
            atoms_feature_all.append(torch.from_numpy(atom_feature))

        feature_dim = atoms_feature_all[0].shape[0]

        if feature_dim != expected_feature_dim:
            if feature_dim < expected_feature_dim:
                for i in range(len(atoms_feature_all)):
                    padding = torch.zeros(expected_feature_dim - feature_dim)
                    atoms_feature_all[i] = torch.cat([atoms_feature_all[i], padding])
            else:
                for i in range(len(atoms_feature_all)):
                    atoms_feature_all[i] = atoms_feature_all[i][:expected_feature_dim]

        g.ndata["atom"] = torch.stack(atoms_feature_all)

        src_list = []
        dst_list = []
        etype_feature_all = []

        num_bonds = mol.GetNumBonds()
        for i in range(num_bonds):
            bond = mol.GetBondWithIdx(i)
            etype_feature = etype_features(bond, use_chirality=True, atompair=True)
            u = bond.GetBeginAtomIdx()
            v = bond.GetEndAtomIdx()
            src_list.extend([u, v])
            dst_list.extend([v, u])
            etype_feature_all.extend([etype_feature, etype_feature])

        if src_list:
            g.add_edges(src_list, dst_list)

            normal_all = []
            total_edges = len(etype_feature_all)
            for et in etype_feature_all:
                normal = etype_feature_all.count(et) / total_edges
                normal = round(normal, 1)
                normal_all.append(normal)

            g.edata["etype"] = torch.tensor(etype_feature_all, dtype=torch.long)
            g.edata["normal"] = torch.tensor(normal_all, dtype=torch.float)
        else:
            g.edata["etype"] = torch.tensor([], dtype=torch.long)
            g.edata["normal"] = torch.tensor([], dtype=torch.float)

        return g

    except Exception as e:
        print(f"Failed to construct molecular graph: {e}")
        raise


# ==============================
# Read all SMILES from CSV
# ==============================
def read_smiles_from_csv(csv_path, smiles_col=None):
    print("\n1. Reading data...")
    df = pd.read_csv(csv_path)
    print(f"CSV shape: {df.shape}")

    if smiles_col is not None:
        if smiles_col not in df.columns:
            raise ValueError(f"The specified smiles_col='{smiles_col}' is not in the CSV columns. Available columns: {list(df.columns)}")
        smiles_series = df[smiles_col]
        used_col = smiles_col
    else:
        candidate_cols = [
            'SMILES', 'smiles', 'Smiles', 'canonical_smiles',
            'Canonical_Smiles', 'CANONICAL_SMILES', 'mol_smiles'
        ]
        used_col = None
        for col in candidate_cols:
            if col in df.columns:
                used_col = col
                smiles_series = df[col]
                break

        if used_col is None:
            used_col = df.columns[0]
            smiles_series = df.iloc[:, 0]

    smiles_series = smiles_series.dropna().astype(str).str.strip()
    smiles_series = smiles_series[smiles_series != ""]
    smiles_list = smiles_series.tolist()

    print(f"Using column: {used_col}")
    print(f"Read {len(smiles_list)} valid SMILES (all will be included in the analysis)")

    if len(smiles_list) == 0:
        raise ValueError("No valid SMILES were read from the CSV.")

    return smiles_list, df, used_col


# ==============================
# Visualization function
# ==============================
def weight_visulize_py2_custom(smiles, atom_weight, save_path, task_name="Task", size=(500, 500)):
    """Visualize atom weights using continuous color mapping only."""
    if not RDKIT_AVAILABLE:
        print("Warning: RDKit is unavailable; visualization cannot be generated")
        return None

    try:
        if hasattr(atom_weight, 'cpu'):
            atom_weight_list = atom_weight.squeeze().cpu().numpy()
        else:
            atom_weight_list = atom_weight.squeeze().numpy()

        if np.ndim(atom_weight_list) == 0:
            atom_new_weight = np.array([float(atom_weight_list)], dtype=float)
        else:
            atom_new_weight = np.asarray(atom_weight_list, dtype=float).flatten()

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"Warning: unable to create a molecule from SMILES '{smiles}'")
            return None

        mol = Chem.AddHs(mol)
        rdDepictor.Compute2DCoords(mol)

        if len(atom_new_weight) > 0:
            weight_min = float(np.min(atom_new_weight))
            weight_max = float(np.max(atom_new_weight))
            if weight_max > weight_min:
                atom_new_weight = (atom_new_weight - weight_min) / (weight_max - weight_min)
            else:
                atom_new_weight = np.full_like(atom_new_weight, 0.5, dtype=float)

        if mol.GetNumAtoms() > 0 and len(atom_new_weight) != mol.GetNumAtoms():
            print(
                f"Warning: the number of atom weights ({len(atom_new_weight)}) does not match "
                f"the number of atoms ({mol.GetNumAtoms()}); using a neutral weight of 0.5."
            )
            atom_new_weight = np.full(mol.GetNumAtoms(), 0.5, dtype=float)

        norm = matplotlib.colors.Normalize(vmin=0, vmax=1)
        cmap = cm.get_cmap('Oranges')
        plt_colors = cm.ScalarMappable(norm=norm, cmap=cmap)

        atom_colors = {i: plt_colors.to_rgba(float(atom_new_weight[i])) for i in range(mol.GetNumAtoms())}
        bond_colors = {}
        for i in range(mol.GetNumBonds()):
            bond = mol.GetBondWithIdx(i)
            u = bond.GetBeginAtomIdx()
            v = bond.GetEndAtomIdx()
            if u < len(atom_new_weight) and v < len(atom_new_weight):
                bond_weight = (atom_new_weight[u] + atom_new_weight[v]) / 2
                bond_colors[i] = plt_colors.to_rgba(float(abs(bond_weight)))
            else:
                bond_colors[i] = plt_colors.to_rgba(0.0)

        drawer = rdMolDraw2D.MolDraw2DSVG(size[0], size[1])
        drawer.SetFontSize(1)
        mol = rdMolDraw2D.PrepareMolForDrawing(mol)
        drawer.DrawMolecule(
            mol,
            highlightAtoms=list(range(mol.GetNumAtoms())),
            highlightBonds=list(range(mol.GetNumBonds())),
            highlightAtomColors=atom_colors,
            highlightBondColors=bond_colors
        )
        drawer.FinishDrawing()

        svg = drawer.GetDrawingText()
        svg_path = f"{save_path}.svg"
        with open(svg_path, 'w', encoding='utf-8') as f:
            f.write(svg)

        print(f"Atom-weight visualization saved to: {svg_path}")
        return svg_path

    except Exception as e:
        print(f"Visualization generation failed: {e}")
        import traceback
        traceback.print_exc()
        return None


# ==============================
# Construct molecular dataset
# ==============================
def create_molecule_dataset(smiles_list):
    print(f"\n2. Constructing molecular graph dataset...")
    print(f"Processing {len(smiles_list)} molecules...")

    dataset = []
    failed_smiles = []

    for i, smiles in enumerate(tqdm(smiles_list, desc="Constructing molecular graphs")):
        try:
            g = construct_RGCN_bigraph_from_smiles(smiles)
            labels = [0.0] * n_tasks
            mask = [1.0] * n_tasks
            dataset.append((smiles, g, labels, mask))
        except Exception as e:
            print(f"Molecule {i + 1}: {smiles} conversion failed - {e}")
            failed_smiles.append(smiles)

    print(f"Successfully processed {len(dataset)} molecules; failed {len(failed_smiles)} ")
    if failed_smiles:
        print("Failed SMILES (first 10):", failed_smiles[:10])

    return dataset, failed_smiles


# ==============================
# Prediction + atom-weight extraction + visualization
# ==============================
def predict_with_atom_weights(model, data_loader, output_dir='./prediction_results',
                              task_names=None, batch_size=32, visualize_all_tasks=False,
                              specific_task_indices=None, visualize_method="continuous"):
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'visualizations'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'atom_weights'), exist_ok=True)

    model.eval()
    all_predictions = []
    all_smiles = []
    all_atom_weights = []

    print(f"Starting prediction; results will be saved to: {output_dir}")
    print(f"Visualization method: {visualize_method}")

    batch_counter = 0

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(data_loader, desc="Prediction progress")):
            smiles, bg, labels, mask = batch
            bg = bg.to('cpu')

            try:
                atom_feats = bg.ndata.pop('atom').float().to('cpu')
            except KeyError:
                if 'h' in bg.ndata:
                    atom_feats = bg.ndata.pop('h').float().to('cpu')
                elif 'feat' in bg.ndata:
                    atom_feats = bg.ndata.pop('feat').float().to('cpu')
                else:
                    raise ValueError("Atom features not found")

            try:
                bond_feats = bg.edata.pop('etype').long().to('cpu')
            except KeyError:
                if 'bond_type' in bg.edata:
                    bond_feats = bg.edata.pop('bond_type').long().to('cpu')
                else:
                    bond_feats = torch.zeros(bg.num_edges(), dtype=torch.long).to('cpu')

            try:
                print(f"\nBatch {batch_idx + 1}: running model prediction...")
                print(f"Input: atom feature shape {atom_feats.shape}, bond feature shape {bond_feats.shape}")

                predictions, atom_weight_list, node_feats = model(
                    bg, atom_feats, bond_feats, norm=None
                )

                print(f"Prediction shape: {predictions.shape}")
                print(f"Length of atom-weight list: {len(atom_weight_list)}")
                print(f"Node feature shape: {node_feats.shape}")

                if isinstance(atom_weight_list, list) and len(atom_weight_list) > 0:
                    print(f"Atom-weight shape for the first task: {atom_weight_list[0].shape}")
                    print(f"Atom-weight range for the first task: [{atom_weight_list[0].min():.4f}, {atom_weight_list[0].max():.4f}]")

            except Exception as e:
                print(f"Batch {batch_idx + 1}: model prediction failed - {e}")
                import traceback
                traceback.print_exc()
                raise

            classification_num = args.get('classification_num', 6)

            if predictions.shape[1] == n_tasks:
                predictions_c = torch.sigmoid(predictions[:, :classification_num])
                predictions_r = predictions[:, classification_num:]
                batch_pred = torch.cat([predictions_c, predictions_r], dim=1)
            else:
                batch_pred = predictions

            unbatch_bg = dgl.unbatch(bg)

            for mol_idx in range(len(smiles)):
                mol_smiles = smiles[mol_idx]
                mol_graph = unbatch_bg[mol_idx]
                mol_atom_count = mol_graph.number_of_nodes()

                start_idx = 0
                for i in range(mol_idx):
                    start_idx += unbatch_bg[i].number_of_nodes()
                end_idx = start_idx + mol_atom_count

                mol_atom_weights_by_task = []
                for task_idx in range(len(task_names)):
                    if task_idx < len(atom_weight_list):
                        task_weights = atom_weight_list[task_idx][start_idx:end_idx]
                        mol_atom_weights_by_task.append(task_weights)
                    else:
                        mol_atom_weights_by_task.append(torch.ones(mol_atom_count) * 0.5)

                all_smiles.append(mol_smiles)
                all_predictions.append(batch_pred[mol_idx].cpu().numpy())
                all_atom_weights.append(mol_atom_weights_by_task)

                if RDKIT_AVAILABLE and (visualize_all_tasks or specific_task_indices is not None):
                    if visualize_all_tasks:
                        tasks_to_visualize = list(range(len(task_names)))
                    else:
                        tasks_to_visualize = specific_task_indices if specific_task_indices is not None else []

                    for task_idx in tasks_to_visualize:
                        if task_idx < len(mol_atom_weights_by_task):
                            task_weights = mol_atom_weights_by_task[task_idx]
                            task_name = task_names[task_idx]

                            if hasattr(task_weights, 'cpu'):
                                weights_np = task_weights.cpu().numpy()
                            else:
                                weights_np = task_weights.numpy()

                            if len(weights_np) > 0:
                                weight_df = pd.DataFrame({
                                    'atom_index': range(len(weights_np)),
                                    'weight': weights_np.flatten()
                                })

                                weight_path = os.path.join(
                                    output_dir, 'atom_weights',
                                    f'mol{batch_counter:06d}_{task_name}_weights.csv'
                                )
                                weight_df.to_csv(weight_path, index=False)

                                save_path = os.path.join(
                                    output_dir, 'visualizations',
                                    f'mol{batch_counter:06d}_{task_name}_{visualize_method}'
                                )
                                weight_visulize_py2_custom(
                                    mol_smiles,
                                    task_weights,
                                    save_path,
                                    task_name=task_name,
                                    )

                batch_counter += 1

    results_df = None
    if len(all_predictions) > 0:
        results_df = pd.DataFrame()
        results_df['SMILES'] = all_smiles

        for task_idx, task_name in enumerate(task_names):
            if task_idx < len(all_predictions[0]):
                results_df[task_name] = [pred[task_idx] for pred in all_predictions]

        csv_path = os.path.join(output_dir, 'predictions.csv')
        results_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f"\nAll prediction results saved to: {csv_path}")

        summary_path = os.path.join(output_dir, 'prediction_summary.txt')
        with open(summary_path, 'w', encoding='utf-8') as f:
            f.write("Prediction summary\n")
            f.write("==============\n")
            f.write(f"Total number of molecules: {len(all_smiles)}\n")
            f.write(f"Number of tasks: {len(task_names)}\n")
            f.write(f"Classification tasks: {task_names[:args.get('classification_num', 6)]}\n")
            f.write(f"Regression tasks: {task_names[args.get('classification_num', 6):]}\n")
            f.write(f"Input feature dimension: {args.get('in_feats', 'unknown')}\n")
            f.write(f"Output directory: {output_dir}\n")
            f.write(f"Visualization method: {visualize_method}\n")
            if specific_task_indices is not None:
                f.write(f"Visualization tasks: {[task_names[i] for i in specific_task_indices]}\n")

        print(f"Summary information saved to: {summary_path}")

        print("\nPreview of prediction results for the first five molecules:")
        for i in range(min(5, len(all_smiles))):
            print(f"\nMolecule {i + 1}: {all_smiles[i]}")
            for task_idx, task_name in enumerate(task_names):
                if task_idx < len(all_predictions[i]):
                    print(f"  {task_name}: {all_predictions[i][task_idx]:.6f}")

    return {
        'smiles': all_smiles,
        'predictions': all_predictions,
        'atom_weights': all_atom_weights,
        'results_df': results_df
    }


# ==============================
# Main program
# ==============================
def main():
    print("=" * 70)
    print("Multi-task model prediction and atom-weight visualization tool")
    print("=" * 70)

    visualize_method = "continuous"
    print(f"\nUsing visualization method: {visualize_method}")

    # 1. Read all SMILES
    try:
        new_data_smiles, raw_df, used_col = read_smiles_from_csv(data_path, smiles_col=smiles_col)
        print(f"Successfully read all SMILES from {data_path} successfully read all SMILES")
    except Exception as e:
        print(f"Failed to read data: {e}")
        raise

    # 2. Constructing molecular graph dataset
    dataset, failed_smiles = create_molecule_dataset(new_data_smiles)

    if len(dataset) == 0:
        print("Error: no molecular graphs were successfully constructed")
        return

    # Save failed molecules
    if len(failed_smiles) > 0:
        failed_path = f'failed_smiles_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
        pd.DataFrame({'failed_smiles': failed_smiles}).to_csv(failed_path, index=False, encoding='utf-8-sig')
        print(f"Failed SMILES saved to: {failed_path}")

    # 3. Create data loader
    print("\n3. Create data loader...")
    batch_size = min(32, len(dataset))
    data_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        collate_fn=collate_molgraphs,
        shuffle=False
    )
    print(f"batch_size = {batch_size}")

    # 4. Set visualization tasks
    print("\n4. Set visualization options...")
    visualize_all_tasks = False

    specific_task_indices = [task_names.index(t) for t in target_task_names if t in task_names]
    if len(specific_task_indices) == 0:
        raise ValueError(f"Target task not found in task_names: {target_task_names}\nCurrent task_names = {task_names}")

    print(f"Target task names: {target_task_names}")
    print(f"Target task indices: {specific_task_indices}")
    print(f"Actual visualization tasks: {[task_names[i] for i in specific_task_indices]}")

    # 5. Prediction and visualization
    print("\n5. Starting prediction and visualization...")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"./prediction_results_{timestamp}"

    results = predict_with_atom_weights(
        model=model,
        data_loader=data_loader,
        output_dir=output_dir,
        task_names=task_names,
        batch_size=batch_size,
        visualize_all_tasks=visualize_all_tasks,
        specific_task_indices=specific_task_indices,
        visualize_method=visualize_method
    )

    print("\n" + "=" * 70)
    print("Prediction completed!")
    print(f"Results saved in: {output_dir}")
    print(f"Number of successfully predicted molecules: {len(results['smiles'])}")
    print(f"Visualization tasks: {target_task_names}")
    if failed_smiles:
        print(f"Number of failed molecules: {len(failed_smiles)}")
    print("=" * 70)


if __name__ == "__main__":
    main()
